# Homelessness

## Step 01 — Prepare the working environment



In [ ]:
# Step 01: Prepare the working environment
# Load the tools needed for tables, regression and maps. Later cells reuse these imports and
# plotting settings.

from pathlib import Path
Path('outputs').mkdir(exist_ok=True)
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler

# Set a consistent appearance for the analysis plots.
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Setup complete: libraries imported successfully.')


## Step 02 — Load the prepared dataset

**1. The file we load**

- we load `data/Final_Cleaned_Dataset_for_Homelessness_Modified.csv`.
- Each row represents one Continuum of Care (CoC) area. The file has **141 rows and 42 columns**.


**3. What you changed while preparing it**

- You combined eight separate population columns—ages **50–54, 55–59, 60–64, 65–69, 70–74, 75–79, 80–84, and 85+**—into one column called **`2024 Population Age 50+`**. 
- You separately removed **`2020 HHs w/Pop 65+`**, which counts households with someone aged 65 or older. Because this measures households rather than people, it was not included in the population sum. The files do not record the reason for this separate removal.
- These changes reduced the file from **50 to 42 columns**: 50 − 8 separate age columns + 1 combined age column − 1 household column = 42.
- The file before these changes is retained at `data/reference/Final_Cleaned_Dataset_for_Homelessness.csv`, so we can compare the earlier and current column arrangements.


**5. What the code checks after loading**

- Count the rows and columns to confirm the size of the loaded table.
- Check whether any CoC code appears more than once.
- Check whether any overall homelessness count is missing.
- Display the first three rows so we can see how the data are arranged.
- For the current file, expect **141 rows, 42 columns, no duplicate CoC codes, and no missing overall homelessness counts**. These are basic file checks, not proof that every source value is complete or correct.


In [ ]:
# Step 02: Read the current enriched input
# Read the area-characteristics file into a table. We need to know which file and rows are being analyzed.

import pandas as pd
enriched_path = 'data/Final_Cleaned_Dataset_for_Homelessness_Modified.csv'

# Load the current file and check its dimensions, CoC keys and outcome completeness.
df_enriched = pd.read_csv(enriched_path)
print('Loaded:', enriched_path)
print('Shape:', df_enriched.shape)
print('Columns:', len(df_enriched.columns))
print('Duplicate COCNUM:', df_enriched['COCNUM'].duplicated().sum())
print('Missing Overall Homeless:', df_enriched['Overall Homeless, 2021'].isna().sum())
display(df_enriched.head(3))


## Step 03 — Count fields and geographic areas

- Read 34 contextual fields, 8 non-contextual fields and 17 jurisdictions including DC. The inventory gives field names, not verified source definitions.

In [ ]:
# Step 03: Count fields and geographic areas
# Separate potential predictors from labels, outcomes and coordinates. A total column count is
# different from a predictor count.

# Keep identifiers, the homelessness outcome and coordinates outside the candidate list.
# The remaining 42 - 8 = 34 columns describe area characteristics.
non_context = ['OBJECTID', 'COCNUM', 'COCNAME', 'ST', 'STATE_NAME', 'Overall Homeless, 2021', 'x', 'y']
context_fields = [c for c in df_enriched if c not in non_context]
display(pd.DataFrame({'Item': ['CoCs',
                'Columns',
                'Contextual indicators',
                'Other fields',
                'Jurisdictions'],
            'Count': [len(df_enriched),
                len(df_enriched.columns),
                len(context_fields),
                len(non_context),
                df_enriched.ST.nunique()]}))

# Count study areas by state/district and list the actual contextual fields.
jurisdictions = df_enriched.groupby(['ST', 'STATE_NAME']).size().rename('CoCs').reset_index()
display(jurisdictions)
display(pd.DataFrame({'Current contextual field': context_fields}))
assert jurisdictions.CoCs.sum() == len(df_enriched)


### PIT Dataset: cleaning the data

- You’ll see these prefixes repeatedly:

- Sheltered ES ... = Emergency Shelter

- Sheltered TH ... = Transitional Housing

- Sheltered SH ... = Safe Haven

- Sheltered Total ... = Sum across sheltered categories

- Unsheltered ... = not in shelter

- So for example:

- Sheltered ES Homeless, 2021 Sheltered TH Homeless, 2021 Sheltered Total Homeless, 2021 Unsheltered Homeless, 2021

## Step 04 — Clean the PIT outcome file

**1. What this step does**

- Prepare homelessness counts for matching.


**3. Process, in order**

1. Standardize column names and convert count fields to numbers
2. Remove AK/HI and missing keys; handle duplicate codes if present
3. Remove empty, constant and mostly-zero PIT columns; check count totals; save the result

**4. How to read the output**

- The supplied run loads 392 PIT rows and retains 386 unique CoCs after geographic exclusions, with 300 cleaned columns. These PIT subcounts are outcomes, not contextual predictors.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 04: Clean the PIT outcome file
# Prepare homelessness counts for matching. Text-formatted numbers and empty fields can prevent
# correct arithmetic.

import pandas as pd
import numpy as np
pit_path = 'data/PIT_dataset - 2021.csv'
df_pit_raw = pd.read_csv(pit_path, dtype=str)
print('Raw shape:', df_pit_raw.shape)

# Remove extra spaces around column names, then make a working copy.
# Example: " CoC Number " becomes "CoC Number".
df_pit_raw.columns = [c.strip() for c in df_pit_raw.columns]
df_pit = df_pit_raw.copy()

# Remove Alaska and Hawaii rows
if 'CoC Number' in df_pit.columns:
    df_pit = df_pit[~df_pit['CoC Number'].astype(str).str.startswith(('AK-', 'HI-'), na=False)]

# Remove rows without a CoC code
df_pit = df_pit.dropna(subset=['CoC Number'])

# Identify columns that should stay as text
id_cols = ['CoC Number', 'CoC Name', 'CoC Category', 'Type of Count']

# Go through the count columns, keeping the four descriptive columns as text.
for col in df_pit.columns:
    if col not in id_cols:

        # Remove commas so text such as "5,111" becomes "5111".
        # Empty strings, "nan" and "None" become missing values, not zeros.
        df_pit[col] = df_pit[col].astype(str).str.replace(',',
            '',
            regex=False).replace({'': np.nan,
                'nan': np.nan,
                'None': np.nan})

        # Convert cleaned text to numbers: "5111" -> 5111.
        # errors="coerce" marks unconvertible text such as "unknown" as missing.
        df_pit[col] = pd.to_numeric(df_pit[col], errors='coerce')

# Store the column name we use later for the main homelessness outcome.
target_col = 'Overall Homeless, 2021'

# If a CoC appears twice, keep its row with the largest overall count.
# This is the existing duplicate rule; the current supplied CoC records are unique.
if target_col in df_pit.columns:
    df_pit = df_pit.sort_values(target_col,
        ascending=False,
        na_position='last').drop_duplicates(subset=['CoC Number'],
        keep='first')
else:
    df_pit = df_pit.drop_duplicates(subset=['CoC Number'], keep='first')

# Find PIT columns that are empty, have only one recorded value, or are at least 95% zero/missing.
# fillna(0) treats missing entries as zero for this screening calculation.
all_empty_cols = [c for c in df_pit.columns if df_pit[c].isna().all()]
df_pit = df_pit.drop(columns=all_empty_cols)
numeric_cols = [c for c in df_pit.columns if c not in id_cols and pd.api.types.is_numeric_dtype(df_pit[c])]
constant_cols = [c for c in numeric_cols if df_pit[c].nunique(dropna=True) <= 1]
mostly_zero_cols = [c for c in numeric_cols if (df_pit[c].fillna(0) == 0).mean() >= 0.95]
drop_noise = sorted(set(constant_cols + mostly_zero_cols))
df_pit_clean = df_pit.drop(columns=drop_noise)
print('\n--- Cleaning Summary ---')
print('After basic cleaning shape:', df_pit.shape)
print('Final cleaned shape:', df_pit_clean.shape)
print('Removed all-empty columns:', len(all_empty_cols))
print('Removed constant columns:', len(constant_cols))
print('Removed mostly-zero columns:', len(mostly_zero_cols))
print('Duplicate CoC Number remaining:', df_pit_clean['CoC Number'].duplicated().sum())

# List the removed fields by reason, then show each dropped name once in the combined list.
for reason, columns in [
    ('All-empty columns', all_empty_cols),
    ('Constant columns', constant_cols),
    ('At least 95% zero/missing columns', mostly_zero_cols),
]:
    print(f'\n{reason} ({len(columns)}):')
    if columns:
        for column in columns:
            print(f'  - {column}')
    else:
        print('  None')

all_dropped_cols = sorted(set(all_empty_cols + drop_noise))
print(f'\nAll dropped columns, without duplicates ({len(all_dropped_cols)}):')
for column in all_dropped_cols:
    print(f'  - {column}')
if not all_dropped_cols:
    print('  None')


# Check whether overall = sheltered + unsheltered, then save the cleaned table.
# Matching totals checks arithmetic; a sheltered-only count can still omit unsheltered people.
if all((c in df_pit_clean.columns for c in ['Overall Homeless, 2021',
            'Sheltered Total Homeless, 2021',
            'Unsheltered Homeless, 2021'])):
    diff = (df_pit_clean['Overall Homeless, 2021'] - (df_pit_clean['Sheltered Total Homeless, 2021'].fillna(0) + df_pit_clean['Unsheltered Homeless, 2021'].fillna(0))).abs()
    print('Rows where Overall != Sheltered Total + Unsheltered:', int((diff > 0).sum()))
out_path = 'outputs/PIT_dataset_2021_cleaned.csv'
df_pit_clean.to_csv(out_path, index=False)
print(f'\nSaved cleaned PIT file: {out_path}')
display(df_pit_clean.head(3))


## Step 05 — Match the two files by CoC code

1. Normalize both code fields
2. Report overlapping and unmatched codes
3. Join common codes, verify 141 unique study rows, and compare overall counts



In [ ]:
# Step 05: Match the two files by CoC code
# Attach the official PIT outcomes to the correct area characteristics. Matching by row order
# could attach another area's count.

enriched = df_enriched.copy()
pit = df_pit_clean.copy()

# Remove surrounding spaces and use uppercase CoC codes in both files.
# Example: " dc-500 " becomes "DC-500", so the same area can be matched.
enriched['COCNUM'] = enriched['COCNUM'].astype(str).str.strip().str.upper()
pit['CoC Number'] = pit['CoC Number'].astype(str).str.strip().str.upper()

# Compare the CoC-code lists: which areas occur in both files, and which occur in only one?
# It works in three steps:
# 1. enriched['COCNUM'] → selects the column.
# 2. .dropna() → removes missing values.
# 3. set(...) → removes duplicates.
enriched_keys = set(enriched['COCNUM'].dropna())
pit_keys = set(pit['CoC Number'].dropna())
common_keys = enriched_keys.intersection(pit_keys)
only_enriched = enriched_keys - pit_keys
only_pit = pit_keys - enriched_keys
print('Enriched CoCs:', len(enriched_keys))
print('PIT CoCs:', len(pit_keys))
print('Common CoCs:', len(common_keys))
print('Only in enriched:', len(only_enriched))
print('Only in PIT:', len(only_pit))

# Attach each area’s PIT row to its characteristics using its CoC code.
# Example: DC-500 is joined to DC-500 even if the two files have different row orders.
merged_df = pd.merge(enriched,
    pit,
    left_on='COCNUM',
    right_on='CoC Number',
    how='inner',
    suffixes=('_enriched',
        '_pit'))
print('\nMerged shape:', merged_df.shape)
print('Unique merged COCNUM:', merged_df['COCNUM'].nunique())
print('Duplicate merged COCNUM rows:', merged_df['COCNUM'].duplicated().sum())
assert len(merged_df) == 141, f'Expected 141 CoCs, got {len(merged_df)}'
assert merged_df['COCNUM'].is_unique

# Compare supplied overall counts; use official PIT counts as the modeling outcome.
pit_col = 'Overall Homeless, 2021_pit'
enr_col = 'Overall Homeless, 2021_enriched'
assert pit_col in merged_df.columns, 'Missing PIT overall-homeless column after merge'
n_match = int((merged_df[pit_col] == merged_df[enr_col]).sum()) if enr_col in merged_df.columns else -1
print(f'Enriched vs PIT overall-homeless match: {n_match}/{len(merged_df)}')
print('Using official PIT counts as the modeling outcome.')
for code in ['DC-500', 'CT-505']:
    row = merged_df.loc[merged_df['COCNUM'] == code, [pit_col]].iloc[0]
    print(f'  {code}: PIT overall homeless = {int(row[pit_col])}')
display(merged_df.head(3))


## Step 06 — Show matched-count evidence

**1. What this step does**

- Display real records and check homelessness accounting.

**3. Process, in order**

1. Select DC-500, CT-505 and NY-600 as examples
2. Subtract sheltered and unsheltered counts from the overall count
3. Check all study rows for disagreement


In [ ]:
# Step 06: Show matched-count evidence
# Display real records and check homelessness accounting. You should be able to explain where
# each outcome came from.

count_evidence = merged_df[['COCNUM',
        'COCNAME',
        'Overall Homeless, 2021_enriched',
        'Overall Homeless, 2021_pit',
        'Sheltered Total Homeless, 2021',
        'Unsheltered Homeless, 2021']].copy()

# Subtract sheltered and unsheltered counts from overall for each area.
# Example: overall 100, sheltered 80, unsheltered 20 gives 100 - 80 - 20 = 0.
count_evidence['Accounting difference'] = count_evidence['Overall Homeless, 2021_pit'] - count_evidence['Sheltered Total Homeless, 2021'] - count_evidence['Unsheltered Homeless, 2021']
display(count_evidence[count_evidence.COCNUM.isin(['DC-500', 'CT-505', 'NY-600'])])
print('Study accounting mismatches:', count_evidence['Accounting difference'].ne(0).sum())
assert count_evidence['Accounting difference'].eq(0).all()
assert count_evidence['Overall Homeless, 2021_enriched'].eq(count_evidence['Overall Homeless, 2021_pit']).all()


## Step 07 — Apply the agreed manual exclusions

**1. What this step does**

- Remove the two variables chosen during the researcher/professor discussion.

- Following the researcher/professor discussion about collinearity, `2024 Burglary Index` and `2024 Rape Index` are manually excluded from the shared candidate pool. `2024 Total Crime Index` remains eligible; subsequent selection determines whether it stays in each model.

**3. Process, in order**

1. Name Burglary Index and Rape Index as the agreed exclusions
2. Remove the two explicitly named fields
3. Verify that row identities are unchanged and save the checkpoint


In [ ]:
# Step 07: Apply the agreed manual exclusions
# Remove Burglary and Rape Index before either model phase. The professor/researcher
# specification must be applied consistently.

# Name the two variables agreed for manual removal.
MANUAL_EXCLUSIONS = ('2024 Burglary Index', '2024 Rape Index')
drop_cols = list(MANUAL_EXCLUSIONS)
rows_before_exclusion = merged_df['COCNUM'].copy()
present = [c for c in drop_cols if c in merged_df.columns]
missing = [c for c in drop_cols if c not in merged_df.columns]

# Remove columns, verify the CoC rows are unchanged, and save the checkpoint.
merged_df = merged_df.drop(columns=present)
assert merged_df['COCNUM'].equals(rows_before_exclusion)
assert not set(MANUAL_EXCLUSIONS).intersection(merged_df.columns)
if present:
    print('Removed from merged_df:', present)
if missing:
    print('Not found (skipped):', missing)
merged_out = 'outputs/analysis_merged_141cocs.csv'
merged_df.to_csv(merged_out, index=False)
print(f'Saved merged file: {merged_out}')
print('Shape:', merged_df.shape)
print('Unique COCNUM:', merged_df['COCNUM'].nunique())


## Step 08 — Inspect a general numeric preview

**1. What this step does**

- Preview numeric columns after merging.


1. Exclude obvious labels and outcomes
2. Filter numeric columns with excessive missingness or no variation
3. Preview the resulting X and outcome


In [ ]:
# Step 08: Inspect a general numeric preview
# Preview numeric columns after merging. This helps inspect data readiness, but is not the
# actual predictor-selection pool.

import pandas as pd
import numpy as np
df_model = merged_df.copy()
target_col = 'Overall Homeless, 2021_pit'

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# Build a broad numeric preview; the actual enriched-only predictor pool is constructed next.
exclude_cols = {'OBJECTID',
    'COCNUM',
    'COCNAME',
    'ST',
    'STATE_NAME',
    'CoC Number',
    'CoC Name',
    'CoC Category',
    'Type of Count',
    target_col,
    'Overall Homeless, 2021_enriched',
    'x',
    'y'}
exclude_cols = {c for c in exclude_cols if c in df_model.columns}
candidate_cols = [c for c in df_model.columns if c not in exclude_cols]
numeric_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(df_model[c])]
X_raw = df_model[numeric_cols].copy()
y = pd.to_numeric(df_model[target_col], errors='coerce')

# Keep preview columns with at most 30% missing entries and at least two different values.
# Use the same rows for area characteristics (X) and homelessness counts (y).
missing_rate = X_raw.isna().mean()
X_step1 = X_raw.loc[:, missing_rate <= 0.3]
X_step2 = X_step1.loc[:, X_step1.nunique(dropna=True) >= 2]
valid = y.notna()
X = X_step2.loc[valid].copy()
y = y.loc[valid].copy()
print('Target column:', target_col)
print('Numeric predictors (initial):', X_raw.shape[1])
print('After missingness filter:', X_step1.shape[1])
print('After variance filter:', X_step2.shape[1])
print('Final X shape:', X.shape)
print('Final y shape:', y.shape)
display(X.head(3))
display(y.head(3))


## Model 1: Overall Homeless (enriched predictors only)

- **Target:** `Overall Homeless, 2021_pit` (official HUD PIT counts matched by `COCNUM`)

- **Predictors:** enriched socioeconomic/housing variables only (from `data/Final_Cleaned_Dataset_for_Homelessness_Modified.csv`).

- **Excluded:** all PIT homelessness subcount columns (to avoid leakage).


## Step 09 — Build the actual contextual predictor pool

**1. What this step does**

- Create the enriched-only X table and official PIT outcome y.

**2. Why we do it**

- Predicting homelessness from its own subcounts would leak outcome information.

**3. Process, in order**

1. Exclude identifiers, coordinates and supplied outcome
2. Retain numeric enriched fields after manual exclusions
3. Remove fields with at least 90% zeros under the specified rule



In [ ]:
# Step 09: Build the actual contextual predictor pool
# Create the enriched-only X table and official PIT outcome y. Predicting homelessness from its
# own subcounts would leak outcome information.

import pandas as pd
import numpy as np
target_col = 'Overall Homeless, 2021_pit'

# Exclude labels, coordinates, outcome and manually removed fields from actual candidates.
non_predictor_enriched = {'OBJECTID', 'COCNUM', 'COCNAME', 'ST', 'STATE_NAME', 'Overall Homeless, 2021', 'x', 'y'}
assert not set(MANUAL_EXCLUSIONS).intersection(merged_df.columns), 'Run the manual-exclusion cell before building predictors.'
enriched_predictor_cols = [c for c in df_enriched.columns if c not in non_predictor_enriched and c in merged_df.columns]
X_enriched = merged_df[enriched_predictor_cols].copy()
y = pd.to_numeric(merged_df[target_col], errors='coerce')
X_enriched = X_enriched.select_dtypes(include=[np.number])
valid = y.notna()
X_enriched = X_enriched.loc[valid].copy()
y = y.loc[valid].copy()

# Remove a characteristic if at least 90% of its entries are zero or missing.
# Example: 90 zero/missing entries out of 100 would meet this cutoff.
ZERO_FRAC = 0.9
sparse_cols = [c for c in X_enriched.columns if (X_enriched[c].fillna(0) == 0).mean() >= ZERO_FRAC]
if sparse_cols:
    print(f'Dropping {len(sparse_cols)} sparse predictors (>= {ZERO_FRAC:.0%} zeros):')
    for c in sparse_cols:
        print(f'  - {c}')
    X_enriched = X_enriched.drop(columns=sparse_cols)
print('Target:', target_col)
print('Rows:', len(y))
print('Enriched predictors:', X_enriched.shape[1])
print('\nPredictor columns:')
print(list(X_enriched.columns))



## Step 10 — Inspect exclusions before VIF

**1. What this step does**

- Show correlations and sparse-field evidence.

**2. Why we do it**

- The reasons for different removal stages must not be confused.

**3. Process, in order**

1. Calculate correlations for the three crime measures before exclusions
2. Display the removed sparse field's zero fraction and missingness
3. Count missing entries among retained candidates

**4. How to read the output**

- The current candidate table has no missing entries. Calculated crime correlations do not reconstruct the historical manual decision.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 10: Inspect exclusions before VIF
# Show correlations and sparse-field evidence. The reasons for different removal stages must not
# be confused.

# Inspect crime correlations before manual exclusions; this does not choose removals
# automatically.
display(df_enriched[list(MANUAL_EXCLUSIONS) + ['2024 Total Crime Index']].corr())
display(pd.DataFrame([{'Field': c,
                'Zero fraction': df_enriched[c].eq(0).mean(),
                'Missing entries': df_enriched[c].isna().sum()} for c in sparse_cols]))
print('Retained candidate missing entries:', int(X_enriched.isna().sum().sum()))
display(pd.DataFrame({'Stage': ['Contextual input',
                'After manual',
                'After sparse'],
            'Count': [len(context_fields),
                len(enriched_predictor_cols),
                X_enriched.shape[1]]}))


## Step 11 — Draw the correlation heatmap before VIF


In [ ]:
# Step 11: Draw the correlation heatmap before VIF
# Visualize pairwise linear overlap among 31 candidates. The plot helps inspect related
# variables before automatic screening.

import os
import numpy as np
import matplotlib.pyplot as plt
os.makedirs('outputs', exist_ok=True)

# Calculate the pairwise correlation matrix; this cell removes no predictors.
X_corr_before = X_enriched.select_dtypes(include=[np.number]).copy()
corr_before = X_corr_before.corr()
n = corr_before.shape[0]
print(f'Before VIF: {n} predictors → {n}×{n} correlation matrix')

# Draw and save the labeled before-VIF heatmap.
fig_w = max(10, n * 0.35)
fig_h = max(8, n * 0.35)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
im = ax.imshow(corr_before.values, cmap='RdBu_r', vmin=-1, vmax=1)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Pearson correlation')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr_before.columns, rotation=90, fontsize=6)
ax.set_yticklabels(corr_before.columns, fontsize=6)
ax.set_title(f'Correlation heatmap BEFORE VIF ({n} predictors)')
if n <= 20:
    for i in range(n):
        for j in range(n):
            val = corr_before.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=5, color='black')
plt.tight_layout()
plt.savefig('outputs/corr_heatmap_before_vif.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: outputs/corr_heatmap_before_vif.png')


## Step 12 — Remove overlapping predictors using VIF

1. Set `VIF_THRESHOLD = 7`; Phase 2 uses this same setting later
2. Calculate each predictor's VIF from the others
3. Remove the single highest VIF if it exceeds 7
4. Recalculate and repeat until all retained VIFs are at most 7

### Understanding this step with a simple example

**1. Why check for overlapping information?**

- We have 31 possible predictors describing our CoCs.
- Some may repeat information contained in the others.
- Imagine a school table containing total students, boys, and girls. Because boys + girls = total students, the three columns repeat information.
- Our area characteristics can also overlap, even when the relationship is less exact.
- Strong overlap makes it difficult for regression to estimate each predictor’s separate relationship with homelessness. The coefficients can become sensitive to changes in the data or included variables.

**2. Hide one column and try to reconstruct it**

- Suppose we check veteran population. Temporarily hide that column and use the other 30 predictors to estimate its values.
- Compare these estimates with the actual veteran population values.
- Repeat for every predictor. Veteran population is just an example; this check does not automatically remove it.
- During this check, the quantity being predicted is the chosen characteristic, not homelessness.

**3. What does R² mean in this check?**

- **R² tells us how much better an equation estimates the numbers compared with guessing the average for everyone.**
- Use veteran population as an example of a variable we check in Step 12.
- The following numbers are made up to explain R²; they are not results from our dataset.

**First, make a simple guess.**

| CoC | Actual veteran population |
|---|---:|
| A | 100 |
| B | 200 |
| C | 300 |

- The average is **(100 + 200 + 300) / 3 = 200**.
- If we ignore differences between the areas, we could guess 200 veterans for every CoC.

| CoC | Actual | Average guess | How far off? |
|---|---:|---:|---:|
| A | 100 | 200 | 100 |
| B | 200 | 200 | 0 |
| C | 300 | 200 | 100 |

- The average guess gets B right, but misses A and C.

**Next, let the equation use information about each area.**

- Hide the veteran-population column and use the other characteristics to estimate it. Suppose the equation produces these estimates:

| CoC | Actual | Equation’s estimate | How far off? |
|---|---:|---:|---:|
| A | 100 | 150 | 50 |
| B | 200 | 200 | 0 |
| C | 300 | 250 | 50 |

- The equation makes smaller mistakes than the average guess.

**R² measures that improvement.**

- To calculate it, square the mistakes and add them. Squaring prevents positive and negative errors from cancelling and makes larger errors count more strongly.

| Method | Calculation | Total squared error |
|---|---|---:|
| Guess the average | 100² + 0² + 100² | 20,000 |
| Use the equation | 50² + 0² + 50² | 5,000 |

- The equation leaves **5,000 / 20,000 = 25%** of the original squared error.
- It has therefore removed **75%** of that error.
- **R² = 1 − 5,000/20,000 = 0.75.**
- This means: **using the other characteristics reduced squared error by 75% compared with guessing the average.** It does not mean every estimate is 75% accurate.

| R² | What it means |
|---|---|
| 0 | The equation does no better than guessing the average |
| 0.75 | The equation removes 75% of the average guess’s squared error |
| 1 | The equation estimates every value exactly |

**Why does this matter in Step 12?**

- If the other variables estimate veteran population extremely well, they already contain much of the same information.
- A high R² therefore signals strong overlap, which we convert into VIF.
- **The average is only the comparison point. The fitted equation uses each area’s characteristics to give that area its own estimate.**
- `.score(others, y_col)` calculates R² on the same rows used to fit that equation. Here it checks overlap within the dataset.

**4. Convert R² into VIF**

- VIF stands for Variance Inflation Factor. It turns the overlap measured above into a screening score.
- **VIF = 1 / (1 − R²).**
- The following examples show how R² becomes VIF:

| R² | VIF calculation | VIF |
|---|---|---:|
| 0.50 | 1 / (1 − 0.50) | 2 |
| 0.90 | 1 / (1 − 0.90) | 10 |
| 0.99 | 1 / (1 − 0.99) | 100 |

- Higher VIF means stronger overlap. Exact reconstruction gives R² = 1, so the code assigns infinite VIF rather than dividing by zero.

**5. Remove one variable, then check again**

- Our agreed VIF cutoff is 7. This is a screening choice, not a rule that proves a variable is unimportant.
- Suppose we start with these example scores:

| Predictor | VIF before removal | Decision in this round |
|---|---:|---|
| A | 12 | Remove first: highest VIF and above 7 |
| B | 9 | Keep for now; recalculate after removing A |
| C | 3 | Keep for now; recalculate after removing A |

- Remove A, then recalculate B and C. Their overlap may have decreased.
- Suppose their new scores are:

| Remaining predictor | Recalculated VIF | Decision |
|---|---:|---|
| B | 4 | Keep: at most 7 |
| C | 2 | Keep: at most 7 |

- Stop because both remaining predictors meet the cutoff. We do not remove B using its earlier score of 9.
- These scores are illustrative, not actual results from our dataset.
- Repeat until every retained predictor has VIF at most 7.

**6. What the current run produces**

- Start with 31 predictors, remove 24 through this procedure, and retain seven for the next selection step.
- The code shows the removal order and final VIF values.
- A removed predictor may still be related to homelessness. Its removal means it exceeded the overlap rule at the round when it was checked.


In [ ]:
# Step 12: Remove overlapping predictors using VIF
# Reduce the candidate pool using the agreed cutoff. Overlapping predictors can make separate
# coefficients unstable.

# Set the VIF cutoff here; Phase 2 later uses this same setting.
VIF_THRESHOLD = 7

from sklearn.linear_model import LinearRegression
import numpy as np
import pandas as pd

# Predict each characteristic from all the other characteristics to measure overlap.
# Here R² concerns that characteristic, not homelessness. Example: R²=.90 gives VIF=10.
def compute_vif(X_df):
    vifs = []
    Xv = X_df.values.astype(float)
    cols = list(X_df.columns)
    for i in range(Xv.shape[1]):
        y_col = Xv[:, i]
        others = np.delete(Xv, i, axis=1)
        if others.shape[1] == 0:
            vif = 1.0
        else:
            r2 = LinearRegression().fit(others, y_col).score(others, y_col)
            vif = np.inf if r2 >= 1 else 1 / (1 - r2)
        vifs.append(vif)
    return pd.DataFrame({'variable': cols, 'VIF': vifs})
X_vif = X_enriched.copy()
dropped = []
step = 0
print('=' * 60)
print('VIF elimination (one variable per loop)')
print(f'Start with {X_vif.shape[1]} predictors | VIF threshold = {VIF_THRESHOLD}')
print('=' * 60)

# Remove the variable with the largest VIF if it exceeds 7, then calculate all VIFs again.
# Example: VIFs 12, 9, 3 -> remove the 12 first; the remaining VIFs can then change.
while X_vif.shape[1] > 1:
    step += 1
    vif_df = compute_vif(X_vif)
    max_vif = float(vif_df['VIF'].max())
    drop_var = vif_df.sort_values('VIF', ascending=False).iloc[0]['variable']
    print(f'\n--- Loop {step} ---')
    print(f'Predictors currently in model: {X_vif.shape[1]}')
    print(f'Worst predictor this loop: {drop_var}')
    print(f'Its VIF: {max_vif:.4f}')
    if max_vif <= VIF_THRESHOLD:
        print(f'STOP: max VIF ({max_vif:.4f}) <= {VIF_THRESHOLD}. No more drops.')
        break
    dropped.append((drop_var, max_vif))
    X_vif = X_vif.drop(columns=[drop_var])
    print(f'DROPPED: {drop_var} (VIF = {max_vif:.4f})')
    print(f'Remaining predictors ({X_vif.shape[1]})')
print('\n' + '=' * 60)
print(f'Finished after {step} loop(s). Dropped {len(dropped)} variable(s).')
print('=' * 60)
if dropped:
    print('\nDrop order (highest VIF removed first each time):')
    for i, (var, vif) in enumerate(dropped, start=1):
        print(f'  {i}. {var}: VIF = {vif:.4f}')
else:
    print('\nNo predictors dropped.')

# Retain the seven-variable pool and show its final VIFs before p-value selection.
X_final = X_vif.copy()
print('\nFinal VIF table:')
display(compute_vif(X_final).sort_values('VIF', ascending=False))
print('\nFinal predictors:', list(X_final.columns))
print('Final predictor count:', X_final.shape[1])


## Step 13 — Display the VIF removal order



In [ ]:
# Step 13: Display the VIF removal order
# Turn the preceding log into a readable table. A reader needs to see which variable left at
# each stage.

vif_removals = pd.DataFrame(dropped, columns=['Removed variable', 'VIF at removal'])
vif_removals.insert(0, 'Step', range(1, len(vif_removals) + 1))
display(vif_removals)


## Step 14 — Draw the correlation heatmap after VIF

In [ ]:
# Step 14: Draw the correlation heatmap after VIF
# Show pairwise overlap within the retained seven-variable pool. The before/after plots help
# explain what VIF screening changed.

import os
import numpy as np
import matplotlib.pyplot as plt
os.makedirs('outputs', exist_ok=True)

# Calculate and draw correlations among the VIF-retained predictors; no further removal occurs
# here.
X_corr_after = X_final.select_dtypes(include=[np.number]).copy()
corr_after = X_corr_after.corr()
n = corr_after.shape[0]
print(f'After VIF: {n} predictors → {n}×{n} correlation matrix')
fig_w = max(8, n * 0.5)
fig_h = max(6, n * 0.5)
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
im = ax.imshow(corr_after.values, cmap='RdBu_r', vmin=-1, vmax=1)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Pearson correlation')
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(corr_after.columns, rotation=90, fontsize=8)
ax.set_yticklabels(corr_after.columns, fontsize=8)
ax.set_title(f'Correlation heatmap AFTER VIF ({n} predictors)')
for i in range(n):
    for j in range(n):
        val = corr_after.values[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color='black')
plt.tight_layout()
plt.savefig('outputs/corr_heatmap_after_vif.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: outputs/corr_heatmap_after_vif.png')
print('\nAfter-VIF predictors:', list(X_corr_after.columns))


## Step 15 — Select and fit the primary OLS model

**1. What this step does**

- Select the four primary predictors and fit one shared equation.

**2. Why we do it**

- After overlap screening we apply the agreed conventional-p selection rule.

**3. Process, in order**

1. Fit the current scaled predictor set
2. Remove the largest p-value above .05 and repeat
3. Scale the final predictors, fit OLS, calculate fit measures and display coefficients

**4. How to read the output**

- Four predictors remain; R²≈.9189, adjusted R²≈.9166 and RMSE≈1,634.79. These are in-sample results with conventional p-values after selection.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.

### A visual walkthrough of Step 15

- **Step 15 builds a homelessness equation, checks the evidence for each predictor’s coefficient, and removes predictors that fail our chosen rule.** Follow the three pictures in order.

**1. First, build the equation**

- Begin with the seven predictors remaining after VIF screening and the recorded homelessness counts for 141 CoCs.
- OLS calculates a weight for each predictor. A weight is called a coefficient.
- It chooses these weights and a starting number (the intercept) to make the total squared prediction error as small as possible.
- Each area’s values go into the same equation, producing its own estimated homelessness count.

<img src="outputs/step15_1_equation.png" alt="How Step 15 builds an equation" width="620" style="max-width: 100%; height: auto;">

- The small equation at the bottom is a made-up example. Our actual equation initially has seven predictors.

**2. Next, check each coefficient using a p-value**

- Suppose we check housing affordability. Temporarily assume its true coefficient is zero after accounting for the other predictors.
- A zero coefficient means that predictor contributes nothing to this equation.
- Even under this assumption, random variation can produce an estimated coefficient different from zero.
- The test compares the size of the estimate with its uncertainty: t = coefficient / standard error. The p-value comes from comparing this t-value with a reference distribution.

<img src="outputs/step15_2_pvalue.png" alt="Visual explanation of the p-value" width="620" style="max-width: 100%; height: auto;">

- **Left:** a result at least this extreme would be fairly common under the zero-coefficient assumption. The p-value is large.
- **Right:** a result at least this extreme would be rare under that assumption. The p-value is small.
- **The orange area is the p-value:** how often a test result at least this extreme would occur if the coefficient were zero and the test assumptions held.
- It is not the probability that the predictor is important or unimportant.
- These two curves are teaching examples, not the actual test results for the removed variables.

**3. Finally, apply our removal rule**

- We chose 0.05 as the cutoff.
- Remove only the predictor with the largest p-value above 0.05, then rebuild the equation.
- Rebuilding matters because changing the included predictors changes the estimated coefficients and their uncertainty.
- The following picture shows the actual sequence in our current run.

<img src="outputs/step15_3_removal.png" alt="Actual Step 15 removal sequence" width="580" style="max-width: 100%; height: auto;">

| Round | Predictor removed | p-value at removal | Predictors remaining |
|---|---|---:|---:|
| 1 | Average Rented Dwellings | 0.4722 | 6 |
| 2 | Healthcare Cost Struggle Index | 0.1351 | 5 |
| 3 | Housing Affordability Index | 0.1083 | 4 |
| 4 | None: largest remaining p-value is 0.0346 | — | 4 |

- After three removals, all four remaining predictors pass the rule. Those four form the global reference equation used later for comparison with GWR.
- The retained predictors are Hispanic Pacific Islander population, civilian veteran population, Total Crime Index, and Tapestry Segment 12D population.

**4. Why remove another three after VIF?**

| Check | Question it answers |
|---|---|
| Step 12: VIF | Does this predictor repeat too much information from the other predictors? |
| Step 15: p-value selection | Does its coefficient meet our chosen statistical-evidence rule in this equation? |

- Passing the overlap check does not guarantee passing the p-value rule.
- Removal means a predictor failed this particular rule in the current model, not that it has no real-world importance.
- These are conventional OLS p-values after data-driven selection. Their interpretation depends on the model assumptions; later influence and uncertainty diagnostics assess their sensitivity.


In [ ]:
# Step 15: Select and fit the primary OLS model
# Start with the seven predictors left after VIF. Fit a homelessness equation, remove the
# largest predictor p-value above .05, and fit again. The current run keeps four predictors. Use
# those four to estimate counts and calculate fit and uncertainty.

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant

# Use .05 as the chosen predictor-selection cutoff. This is a modeling rule, not proof of importance.
P_ALPHA = 0.05

# Fit OLS, remove the variable with the largest p-value above .05, and fit again.
# Example: p-values .20, .08, .01 -> remove the .20 variable first, then recalculate.
def backward_pvalue_elimination(X_df, y_series, alpha=0.05):
    """
    Drop predictors ONE AT A TIME:
    each loop fits OLS, finds the single highest p-value,
    drops that one variable if p > alpha, then repeats.
    """

    # Work on copies and record removals so the starting predictor table remains available.
    X_work = X_df.copy()
    y_work = y_series.copy()
    dropped = []
    step = 0
    print('=' * 60)
    print('Backward p-value elimination (one variable per loop)')
    print(f'Start with {X_work.shape[1]} predictors | alpha = {alpha}')
    print('=' * 60)
    while X_work.shape[1] > 1:
        step += 1

        # Scale the variables still being tested to 0–1, then fit OLS for this selection round.
        scaler = MinMaxScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X_work), columns=X_work.columns, index=X_work.index)
        ols = OLS(y_work, add_constant(X_scaled)).fit()

        # Compare each predictor coefficient with zero using its estimated uncertainty.
        # Exclude the intercept from predictor selection; find the largest remaining p-value.
        pvals = ols.pvalues.drop('const')
        worst_var = pvals.idxmax()
        worst_p = float(pvals.max())
        print(f'\n--- Loop {step} ---')
        print(f'Predictors currently in model: {X_work.shape[1]}')
        print(f'Worst predictor this loop: {worst_var}')
        print(f'Its p-value: {worst_p:.6f}')

        # If even the largest predictor p-value is at most .05, every current predictor passes.
        if worst_p <= alpha:
            print(f'STOP: worst p ({worst_p:.6f}) <= {alpha}. No more drops.')
            break

        # Otherwise record and remove this one predictor, then refit on the next loop.
        # Example: .30, .08, .01 -> remove .30 first; the other p-values may change after refitting.
        dropped.append((worst_var, worst_p))
        X_work = X_work.drop(columns=[worst_var])
        print(f'DROPPED: {worst_var} (p = {worst_p:.6f})')
        print(f'Remaining predictors ({X_work.shape[1]}): {list(X_work.columns)}')
    print('\n' + '=' * 60)
    print(f'Finished after {step} loop(s). Dropped {len(dropped)} variable(s).')
    print('=' * 60)
    return (X_work, y_work, dropped)

# Put characteristics and counts in the same table, remove incomplete rows, and run selection.
model_df = pd.concat([X_final, y.rename('y')], axis=1).dropna()
X_candidates = model_df[X_final.columns]
y_candidates = model_df['y']
print(f'Starting predictors (after VIF): {X_candidates.shape[1]}')
X_model, y_model, dropped_predictors = backward_pvalue_elimination(X_candidates, y_candidates, alpha=P_ALPHA)
if dropped_predictors:
    print('\nDrop order (first dropped = least significant at that step):')
    for i, (var, pval) in enumerate(dropped_predictors, start=1):
        print(f'  {i}. {var}: p = {pval:.4f}')
else:
    print('\nNo predictors dropped.')
print(f'\nKept predictors ({X_model.shape[1]}):')
for col in X_model.columns:
    print(f'  - {col}')

# Store the selected names, scale their inputs and fit the final shared equation.
# Scaling: minimum -> 0, maximum -> 1. The outcome remains a count of people.
MODEL1_PREDICTORS = list(X_model.columns)
scaler = MinMaxScaler()
X_model_scaled = pd.DataFrame(scaler.fit_transform(X_model), columns=X_model.columns, index=X_model.index)
model = LinearRegression()
model.fit(X_model_scaled, y_model)

# Apply the final equation to the same CoCs used for fitting to obtain their estimated counts.
y_pred = model.predict(X_model_scaled)
n = len(y_model)
p = X_model_scaled.shape[1]

# Measure improvement over guessing the average: R² = 1 - model squared error / mean-guess squared error.
# Higher means better in-sample fit; it does not mean every prediction has that percentage accuracy.
r2 = r2_score(y_model, y_pred)

# Square each count error and average: MSE. Take its square root for RMSE in people.
# Lower means smaller errors; squaring gives large mistakes greater weight.
mse = mean_squared_error(y_model, y_pred)
rmse = np.sqrt(mse)

# Adjust R² for sample size and predictor count so adding predictors is not rewarded automatically.
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print('\n' + '=' * 55)
print('MODEL 1: Overall Homeless, 2021')
print('Predictors: VIF + p-value pruning (min-max scaled)')
print('Evaluation: IN-SAMPLE (no train-test split)')
print('=' * 55)
print(f'Observations (n): {n}')
print(f'Predictors (p):   {p}')
print(f'R-squared:        {r2:.4f}')
print(f'Adjusted R²:      {adj_r2:.4f}')
print(f'MSE:              {mse:,.2f}')
print(f'RMSE:             {rmse:,.2f}')
print('=' * 55)

# Pair each predictor name with its equation weight; sort by absolute size for display.
# Predictor units differ even after scaling, so size alone does not establish policy importance.
coef_df = pd.DataFrame({'variable': X_model_scaled.columns,
        'coefficient': model.coef_}).sort_values('coefficient',
    key=abs,
    ascending=False)
print(f'\nIntercept: {model.intercept_:,.4f}')
display(coef_df)

# Fit the same final equation with statsmodels to obtain standard errors, t-values and p-values.
ols_final = OLS(y_model, add_constant(X_model_scaled)).fit()
coef_table = pd.DataFrame({'variable': ['intercept'] + list(X_model.columns),
        'coefficient': ols_final.params.values,
        'std_err': ols_final.bse.values,
        't_stat': ols_final.tvalues.values,
        'p_value': ols_final.pvalues.values}).sort_values('p_value')
print('\nFinal model: coefficients and p-values')
display(coef_table)


## Step 16 — Display the complete selection ledger


In [ ]:
# Step 16: Display the complete selection ledger
# Build a record for all 34 original characteristics showing why each was removed or kept. This
# lets us trace the complete selection process. Also check overlap again among the four final
# predictors.

vif_map = dict(dropped)
p_map = dict(dropped_predictors)

# For every original characteristic, show where it was removed or whether it stayed in the model.
selection_ledger = pd.DataFrame([{'Field': c,
            'Stage': 'Manual exclusion' if c in MANUAL_EXCLUSIONS else 'Sparse exclusion' if c in sparse_cols else 'VIF exclusion' if c in vif_map else 'P-value exclusion' if c in p_map else 'Retained primary predictor',
            'VIF at removal': vif_map.get(c,
                np.nan),
            'p at removal': p_map.get(c,
                np.nan)} for c in context_fields])
display(selection_ledger)
display(pd.DataFrame({'Stage': ['Input contextual',
                'After manual',
                'After sparse',
                'After VIF',
                'After p screen'],
            'Count': [len(context_fields),
                len(enriched_predictor_cols),
                X_enriched.shape[1],
                X_final.shape[1],
                len(MODEL1_PREDICTORS)]}))

# Check VIF again on the final four predictors.
display(compute_vif(X_model))


## Step 17 — Work through a prediction and R²

**1. What this step does**

- Show actual arithmetic behind the primary model's outputs.

**2. Why we do it**

- A reader should be able to explain prediction, error and fit separately.

**3. Process, in order**

1. Show predictor minima/maxima and scaled DC inputs
2. Multiply each input by its coefficient and add the intercept
3. Calculate residuals, SSE, SST, R², adjusted R² and RMSE

**4. How to read the output**

- DC-500 has its own prediction, not the sample mean. R² compares squared errors against the mean baseline; a scaled coefficient describes a minimum-to-maximum contrast.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 17: Work through a prediction and R²
# Work through DC-500’s actual prediction so the equation is visible as arithmetic. Then
# calculate R², adjusted R² and RMSE from all prediction errors, and keep the errors beside
# their CoC codes.

# Show each predictor’s observed minimum/maximum and the coefficient for its scaled input.
display(pd.DataFrame({'Predictor': MODEL1_PREDICTORS,
            'Minimum': scaler.data_min_,
            'Maximum': scaler.data_max_,
            'Scaled coefficient': model.coef_}))

# For DC-500, multiply each scaled input by its coefficient, then add the intercept.
# Example: an input of .2 and coefficient 100 contribute .2 * 100 = 20 to the prediction.
dc_index = merged_df.index[merged_df.COCNUM.eq('DC-500')][0]
contributions = pd.DataFrame({'Predictor': MODEL1_PREDICTORS,
        'Raw DC input': X_model.loc[dc_index].values,
        'Scaled DC input': X_model_scaled.loc[dc_index].values,
        'Coefficient': model.coef_})
contributions['Contribution'] = contributions['Scaled DC input'] * contributions.Coefficient
display(contributions)
dc_prediction = model.intercept_ + contributions.Contribution.sum()
print('DC-500 intercept:',
    model.intercept_,
    'prediction:',
    dc_prediction,
    'observed:',
    y_model.loc[dc_index],
    'residual:',
    y_model.loc[dc_index] - dc_prediction)
np.testing.assert_allclose(dc_prediction, model.predict(X_model_scaled.loc[[dc_index]])[0])

# Compare squared model errors (SSE) with squared errors from predicting the mean (SST).
# Example: SSE=20 and SST=100 give R² = 1 - 20/100 = .80. RMSE stays in people.
sse = float(np.sum((y_model.to_numpy() - y_pred) ** 2))
sst = float(np.sum((y_model.to_numpy() - y_model.mean()) ** 2))
display(pd.DataFrame({'Quantity': ['n',
                'p',
                'Mean baseline',
                'SSE',
                'SST',
                'R²',
                'Adjusted R²',
                'RMSE'],
            'Value': [n,
                p,
                y_model.mean(),
                sse,
                sst,
                1 - sse / sst,
                1 - sse / sst * (n - 1) / (n - p - 1),
                np.sqrt(sse / n)]}))
np.testing.assert_allclose(1 - sse / sst, r2)

# Keep counts and residuals attached to CoC identities for later diagnostics.
prediction_evidence = pd.DataFrame({'COCNUM': merged_df.loc[y_model.index,
            'COCNUM'].values,
        'COCNAME': merged_df.loc[y_model.index,
            'COCNAME'].values,
        'Observed': y_model.values,
        'OLS prediction': y_pred,
        'OLS residual': y_model.values - y_pred})


### Interpretation (Model 1)

- **In-sample** model on 141 CoCs (no train-test split).
- **Outcome:** official PIT overall homeless counts (`Overall Homeless, 2021_pit`).
- Predictors: enriched socioeconomic variables only; PIT subcounts excluded (no leakage).
- Final predictors after manual exclusions, sparse filter, VIF (<=7), and p-value screening.
- Coefficients show **association**, not causation.

- **Phase 2 (next):** optional repeat for other PIT targets (Sheltered / Unsheltered).

## Phase 2: PIT homelessness targets (enriched predictors only)

- Same feature-selection and regression pipeline as Model 1.
- Targets: Sheltered ES, Sheltered TH, Sheltered Total, Unsheltered.
- Predictors: enriched variables only (no PIT subcounts in X).
- In-sample simple linear regression on 141 CoCs.
- Both phases use `VIF_THRESHOLD = 7`, defined in Step 12. Run Step 12 before this phase.
- Burglary and Rape Index exclusions and the sparse filter are already applied to X_enriched.
- Each target uses its own complete rows for subsequent VIF and p-value selection.

## Step 18 — Fit the four additional homelessness outcomes

**1. What this step does**

- Repeat model selection independently for each additional outcome.

**2. Why we do it**

- Different outcomes may require different predictors.

**3. Process, in order**

1. Start each outcome from the same 31 candidates
2. Apply VIF and p-value elimination for that outcome
3. Fit its retained predictors and record n, predictor count, R², adjusted R² and RMSE

**4. How to read the output**

- The final predictor counts are 5, 3, 4 and 3. Do not assume all five outcome models use the primary four predictors.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 18: Fit the four additional homelessness outcomes
# Build a separate equation for each of four additional homelessness outcomes. Each starts with
# the same 31 contextual candidates and applies VIF <= 7 and p <= .05 selection independently.
# The primary model’s four predictors are not forced into these fits.

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
from IPython.display import display
import pandas as pd
import numpy as np
P_ALPHA = 0.05

# Define the same VIF-screening and conventional-p selection rules for additional outcomes.
def compute_vif(X_df):
    """VIF among predictors only (target y is NOT used)."""
    vifs = []
    Xv = X_df.values.astype(float)
    cols = list(X_df.columns)
    for i in range(Xv.shape[1]):

        # Temporarily hide one predictor and estimate it from all the others to measure linear overlap.
        y_col = Xv[:, i]
        others = np.delete(Xv, i, axis=1)
        if others.shape[1] == 0:
            vif = 1.0
        else:
            r2 = LinearRegression().fit(others, y_col).score(others, y_col)
            vif = np.inf if r2 >= 1 else 1 / (1 - r2)
        vifs.append(vif)
    return pd.DataFrame({'variable': cols, 'VIF': vifs})

# Remove the largest VIF above 7 one at a time, recalculating after each removal.
def vif_elimination(X_df, threshold):
    """Backward VIF elimination — same logic as Cell 15."""
    X_work = X_df.copy()
    while X_work.shape[1] > 1:
        vif_df = compute_vif(X_work)
        if vif_df['VIF'].max() <= threshold:
            break
        drop_var = vif_df.sort_values('VIF', ascending=False).iloc[0]['variable']
        X_work = X_work.drop(columns=[drop_var])
    return X_work

# Refit and remove the largest predictor p-value above .05 one at a time.
def backward_pvalue_elimination(X_df, y_series, alpha=0.05):
    """Same as the final primary OLS section."""
    X_work = X_df.copy()
    y_work = y_series.copy()
    dropped = []
    while X_work.shape[1] > 1:
        scaler = MinMaxScaler()
        X_scaled = pd.DataFrame(scaler.fit_transform(X_work), columns=X_work.columns, index=X_work.index)
        ols = OLS(y_work, add_constant(X_scaled)).fit()

        # Check predictor p-values, leaving the intercept out of the removal candidates.
        pvals = ols.pvalues.drop('const')
        worst_var = pvals.idxmax()
        worst_p = pvals.max()
        if worst_p <= alpha:
            break
        dropped.append((worst_var, worst_p))
        X_work = X_work.drop(columns=[worst_var])
    return (X_work, y_work, dropped)

# Fit one outcome independently from the shared contextual candidate pool.
def run_enriched_regression(target_name, predictor_cols, merged_df):
    """
    Full Model 1 pipeline, applied to one Phase 2 target.
    Uses that target's own valid rows — NOT X_final from Cell 15.
    """

    # Match this outcome with its candidate predictors and keep complete rows for this fit.
    y = pd.to_numeric(merged_df[target_name], errors='coerce')
    X_full = merged_df[predictor_cols].copy()
    model_df = pd.concat([X_full, y.rename('y')], axis=1).dropna()
    X_raw = model_df[predictor_cols]
    y_raw = model_df['y']

    # First remove excessive overlap, then apply the conventional p-value selection rule.
    X_vif = vif_elimination(X_raw, threshold=VIF_THRESHOLD)
    X_model, y_model, dropped_predictors = backward_pvalue_elimination(X_vif, y_raw, alpha=P_ALPHA)

    # Scale the final selected inputs and fit this outcome’s own equation.
    scaler = MinMaxScaler()
    X_model_scaled = pd.DataFrame(scaler.fit_transform(X_model), columns=X_model.columns, index=X_model.index)
    model = LinearRegression()
    model.fit(X_model_scaled, y_model)
    y_pred = model.predict(X_model_scaled)
    n = len(y_model)
    p = X_model_scaled.shape[1]

    # Evaluate predictions on the fitted rows: larger R² and smaller RMSE indicate closer fit.
    r2 = r2_score(y_model, y_pred)
    mse = mean_squared_error(y_model, y_pred)
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return {'target': target_name,
        'n': n,
        'p': p,
        'vif_predictors': list(X_vif.columns),
        'predictors': list(X_model.columns),
        'dropped_pvalue': dropped_predictors,
        'r2': r2,
        'adj_r2': adj_r2,
        'rmse': np.sqrt(mse)}
predictor_cols = list(X_enriched.columns)
assert not set(MANUAL_EXCLUSIONS).intersection(predictor_cols)

# Run the four additional outcomes and collect their selected predictors and performance.
pit_targets = ['Sheltered ES Homeless, 2021',
    'Sheltered TH Homeless, 2021',
    'Sheltered Total Homeless, 2021',
    'Unsheltered Homeless, 2021']
print('=' * 60)
print('PHASE 2: Same pipeline as Model 1 (independent run per target)')
print(f'Starting enriched predictors: {len(predictor_cols)}')
print(f'VIF threshold: {VIF_THRESHOLD} | P-value alpha: {P_ALPHA}')
print('=' * 60)
results = []
for t in pit_targets:
    if t not in merged_df.columns:
        print(f'MISSING COLUMN: {t}')
        continue
    res = run_enriched_regression(t, predictor_cols, merged_df)
    results.append(res)
    print('\n' + '=' * 60)
    print(res['target'])
    print(f"n = {res['n']}")
    print(f"After VIF ({len(res['vif_predictors'])}): {res['vif_predictors']}")
    if res['dropped_pvalue']:
        print('Dropped by p-value:')
        for var, pval in res['dropped_pvalue']:
            print(f'  - {var}: p = {pval:.4f}')
    else:
        print('Dropped by p-value: none')
    print(f"Final predictors ({res['p']}): {res['predictors']}")
    print(f"R² = {res['r2']:.4f} | Adj R² = {res['adj_r2']:.4f} | RMSE = {res['rmse']:,.2f}")
summary_df = pd.DataFrame([{'target': r['target'],
            'n': r['n'],
            'predictors': r['p'],
            'r2': r['r2'],
            'adj_r2': r['adj_r2'],
            'rmse': r['rmse']} for r in results])
print('\n=== PHASE 2 SUMMARY TABLE ===')
display(summary_df)


## Five-outcome performance table — Corrections 17, 46

- The primary outcome and four additional outcomes each use their own selected predictors.
- Additional models start from the shared 31-variable pool, not the primary final four.
- The preceding code displays their removal decisions.
- Combine all five outputs for the paper table.
- Raw RMSE across differently scaled outcomes is not a model-quality ranking.

## Step 19 — Combine all five model-performance rows

**1. What this step does**

- Build the paper's full outcome-comparison table.

**2. Why we do it**

- The primary and additional results otherwise appear in separate places.

**3. Process, in order**

1. Create the primary row
2. Append the four additional rows
3. Display retained lists and save the combined table

**4. How to read the output**

- Compare fit within each outcome. Raw RMSE depends on the outcome scale and is not a fair standalone ranking across different outcomes.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 19: Combine all five model-performance rows
# Combine the overall model and four additional outcome models into one performance table for
# the paper. Each row identifies its sample size, selected predictor count and fit scores; the
# scores describe the fitted sample.

paper_performance = pd.concat([pd.DataFrame([{'target': 'Overall Homeless, 2021_pit',
                    'n': len(y_model),
                    'predictors': len(MODEL1_PREDICTORS),
                    'r2': r2,
                    'adj_r2': adj_r2,
                    'rmse': rmse}]),
        summary_df],
    ignore_index=True)
display(paper_performance)
display(pd.DataFrame([{'Outcome': r['target'], 'Retained predictors': '; '.join(r['predictors'])} for r in results]))
paper_performance.to_csv('outputs/paper_model_performance.csv', index=False)


## Phase 3: Moran's I (spatial autocorrelation)

- **Question:** Are Model 1 residuals spatially clustered?

- Uses **official HUD CoC polygon boundaries**.
- Joined to the **merged analysis table** on `COCNUM`.
- Alaska/Hawaii excluded (non-contiguous).
- Neighbors: **KNN k=5**.
- Homelessness outcome reference: **PIT overall homeless** (`Overall Homeless, 2021_pit`).

### Understanding the Azimuthal Equidistant projection

- **Azimuthal Equidistant is a way of turning locations on the curved Earth into positions on a flat map.**
- We use it in the spatial analysis because Moran’s I and GWR need distances between CoCs.

**1. Why do we need a projection?**

- Geographic locations initially use longitude and latitude, measured in degrees.
- For this analysis, we want coordinates in metres so we can ask which CoCs are nearest, which are within 50 km, and how much weight a nearby CoC should receive in GWR.
- The projection converts the locations into metre-based coordinates.

**2. What does Azimuthal Equidistant mean?**

- Imagine placing a flat sheet against a globe at a chosen central point. This is a way to picture the projection; its coordinates are calculated mathematically.
- Distances from that central point are preserved.
- Directions from that central point are preserved.
- Distances between other pairs of places can still have some distortion.

**3. Why did we choose it for this project?**

- Our 141 study CoCs extend approximately from Georgia to Maine.
- We placed the projection’s centre near the middle of that region: **39.5° north, 77.5° west**.
- We use WGS84 regional Azimuthal Equidistant coordinates in metres: `+proj=aeqd +lat_0=39.5 +lon_0=-77.5 +datum=WGS84 +units=m`.
- To check distance accuracy, compare distances between the same 141 representative points in three projections with distances along the WGS84 Earth surface. There are 9,870 point pairs; Step 21 shows this calculation.

| Projection | Median absolute distance discrepancy | Largest absolute distance discrepancy |
|---|---:|---:|
| Regional Azimuthal Equidistant | 0.0057% | 0.329% |
| CONUS Albers | 0.448% | 0.975% |
| Web Mercator | 30.46% | 41.04% |

- The median describes the middle percentage discrepancy across all pairs.
- **Regional Azimuthal Equidistant gave the closest distance measurements among these three choices for our study locations.** This does not establish that it is best for every region or every map purpose.

**4. How do we use it here?**

- Transform the HUD CoC polygons into this projection.
- Calculate each polygon’s representative centre point, called a centroid, in the projected coordinates.
- Use distances between those points for Moran neighborhoods, distance-band checks and later GWR geographic weights.
- Keep the projection and its centre fixed so the analysis is reproducible.
- These are straight-line distances between representative points, not road or service-travel distances.
- **The projection helps us measure nearby accurately. It does not itself fit the regression or improve the homelessness data.**

- The image below shows one real CoC pair. It illustrates distance measurement rather than proving that every model result improves.

![Same two CoCs measured on the Earth and in three projections](attachment:projection_comparison_explained.png)

- **What changes:** The regression inputs, 141 CoCs, and VIF/p-value selection stay the same.
- Spatial centroids, distance bands, spatial weights, GWR bandwidth, local coefficients, local significance, Monte Carlo results, and residual Moran's I must all be recalculated from this projection.
- Keep the projection and its center fixed in the study methods.
- This is a location-based measure; it does not measure road or service-travel distance.

### Moran’s I in simple words

- **Moran’s I asks: “Are nearby areas making similar mistakes?”**
- First, our model estimates homelessness for each CoC.
- Compare the estimate with the recorded count. The difference is the model’s error, called a residual: **recorded count − estimated count**.

**1. Imagine three neighboring areas**

- This example is made up to explain the concept; these are not actual CoC results.

| Area | Recorded count | Model estimate | Mistake |
|---|---:|---:|---|
| A | 100 | 80 | 20 too low |
| B | 150 | 120 | 30 too low |
| C | 200 | 175 | 25 too low |

- The model estimates too few people in all three neighboring areas. That is a geographic pattern in its mistakes.
- A positive residual means the estimate is too low. A negative residual means the estimate is too high.

**2. Check this across the study**

- Moran’s I summarizes neighbor comparisons across all 141 CoCs. Our primary test uses five nearest neighbors per CoC.

| Moran’s I | What it suggests |
|---|---|
| Positive | Neighbors tend to have similar mistakes |
| Near zero | No clear overall pattern of similar or opposing mistakes under the chosen neighborhood rule |
| Negative | Neighbors tend to have opposing mistakes: too low beside too high |

**3. Why do we care?**

- A model might fit well overall but repeatedly miss the counts in a particular part of the map.
- Moran’s I helps us see whether the errors are geographically grouped.
- **Moran’s I checks the pattern of the mistakes, rather than how large the mistakes are.**

**4. What happened in our project?**

| Model | Residual Moran’s I | Permutation p-value | Finding under the primary test |
|---|---:|---:|---|
| OLS | 0.1513 | 0.0009 | Evidence of geographic clustering in the errors |
| GWR | 0.0139 | 0.3258 | Much smaller clustering; no statistically supported positive clustering |

- The p-value comes from comparing the real pattern with patterns obtained by randomly rearranging the same residual values among the same locations.
- The GWR result does not prove that every geographic pattern has disappeared or that every prediction is accurate.
- R² measures how well estimates fit the counts; Moran’s I checks the geographic arrangement of the remaining errors.

### Moran's I simulation settings used by this notebook

- The observed Moran's I is calculated from the actual residuals and spatial neighbors.
- Its simulation p-value is estimated by repeatedly permuting the residual values among the same CoC locations.
- The locations and residual values remain the same; only their assignment to locations changes.

- The notebook uses **9,999 permutations** and the base seed **5536** for the Moran's I simulations (OLS: 5536; each distance band: 5536 plus its distance in km; GWR: 105536).
- The seed makes the permutation sequence reproducible.
- The explicit permutation count makes the precision of the reported p-value clear.
- These settings affect the simulated p-value, not the observed Moran's I.

- The same settings are used for the OLS residual KNN test, the distance-band sensitivity tests, and the GWR residual KNN test.
- The spatial-variability Monte Carlo test in Phase 5 is a separate procedure and keeps its own documented 1,000-iteration setting.

## Step 20 — Load and match geographic boundaries


1. Load the supplied shapefile or download it if absent
2. Normalize codes and exclude AK/HI
3. Join study codes and define regional AEQD coordinates in metres




In [ ]:
# Step 20: Load and match geographic boundaries
# Load the HUD polygons defining CoC areas and match them to our table using CoC codes. Check
# that all 141 study areas have geometry. The projection setting provides metre-based locations
# for later spatial calculations.

import os
import urllib.request
import zipfile
import pandas as pd
import geopandas as gpd
DATA_DIR = 'data'
HUD_ZIP = os.path.join(DATA_DIR, 'hud_coc_boundaries.zip')
HUD_EXTRACT = os.path.join(DATA_DIR, 'hud_coc_boundaries')
HUD_URL = 'https://opendata.arcgis.com/api/v3/datasets/c930d736b1764c259371fc7111e02740_0/downloads/data?format=shp&spatialRefId=4326&where=1%3D1'
NON_CONTIGUOUS_STATES = {'AK', 'HI'}
TARGET_COL = 'Overall Homeless, 2021_pit'
ANALYSIS_CRS = '+proj=aeqd +lat_0=39.5 +lon_0=-77.5 +datum=WGS84 +units=m +no_defs +type=crs'

# Use existing source boundaries, obtaining them only when local files are absent.
def ensure_hud_shapefile():
    os.makedirs(DATA_DIR, exist_ok=True)
    os.makedirs(HUD_EXTRACT, exist_ok=True)
    shp_files = [f for f in os.listdir(HUD_EXTRACT) if f.endswith('.shp')]
    if shp_files:
        return os.path.join(HUD_EXTRACT, shp_files[0])
    if not os.path.exists(HUD_ZIP):
        print('Downloading HUD CoC boundaries (~45 MB)...')
        urllib.request.urlretrieve(HUD_URL, HUD_ZIP)
        print('Download complete.')
    with zipfile.ZipFile(HUD_ZIP, 'r') as z:
        z.extractall(HUD_EXTRACT)
    shp_files = [f for f in os.listdir(HUD_EXTRACT) if f.endswith('.shp')]
    if not shp_files:
        raise FileNotFoundError('No .shp found in data/hud_coc_boundaries/')
    return os.path.join(HUD_EXTRACT, shp_files[0])

# Load and normalize the boundary table.
shp_path = ensure_hud_shapefile()
hud = gpd.read_file(shp_path)
hud['COCNUM'] = hud['COCNUM'].astype(str).str.strip().str.upper()
hud = hud[~hud['COCNUM'].str.startswith(('AK-', 'HI-'), na=False)]
hud = hud.drop_duplicates('COCNUM', keep='first')

# Attach the official study counts by CoC code and verify geographic coverage.
tab = merged_df[['COCNUM', 'COCNAME', TARGET_COL]].copy()
tab['COCNUM'] = tab['COCNUM'].astype(str).str.strip().str.upper()
tab['homeless'] = pd.to_numeric(tab[TARGET_COL], errors='coerce')
gdf = hud[['COCNUM', 'geometry']].merge(tab, on='COCNUM', how='inner')
assert len(gdf) == 141, f'Expected 141 spatial CoCs, got {len(gdf)}'
print(f'HUD polygons (AK/HI removed): {len(hud)}')
print(f'Joined spatial rows: {len(gdf)}')
print(f'Source CRS: {gdf.crs}')
print(f'Analysis CRS: {ANALYSIS_CRS}')
display(gdf[['COCNUM', 'COCNAME', 'homeless']].head())


## Step 21 — Check projection accuracy and geometry

**1. What this step does**

- Measure distance distortion and geometry-repair sensitivity.

1. Compute AEQD centroids and WGS84 Earth distances for all pairs
2. Compare the same points in three projections
3. Validate polygons, repair copies and compare centroids




In [ ]:
# Step 21: Check projection accuracy and geometry
# Check how closely flat-map distances match Earth-surface distances between the same points.
# Compare three projections and measure how much repairing invalid polygons moves their
# centroids. These checks assess the locations used in spatial weighting.

from pyproj import Geod
from scipy.spatial.distance import pdist

# Use the same centroid points for every projection comparison.
# Compare flat-map distances with distances along the WGS84 Earth surface.
projected = gdf.to_crs(ANALYSIS_CRS)
points = gpd.GeoSeries(projected.geometry.centroid, index=gdf.index, crs=ANALYSIS_CRS)
ll = points.to_crs(4326)
pi, pj = np.triu_indices(len(gdf), 1)
earth = Geod(ellps='WGS84').inv(ll.x.values[pi], ll.y.values[pi], ll.x.values[pj], ll.y.values[pj])[2]
projection_rows = []

# Calculate distance discrepancies for regional AEQD, Mercator and Albers.
for label, crs in [('Regional AEQD', ANALYSIS_CRS), ('Web Mercator', 'EPSG:3857'), ('CONUS Albers', 'EPSG:5070')]:
    q = points.to_crs(crs)
    flat = pdist(np.column_stack([q.x, q.y]))

    # Express absolute distance differences as percentages of the Earth-surface distances.
    # Example: flat 101 km versus Earth 100 km means a 1% discrepancy.
    error = 100 * np.abs(flat - earth) / earth
    projection_rows.append({'Projection': label,
            'Pairs': len(error),
            'Median absolute discrepancy %': np.median(error),
            'Maximum absolute discrepancy %': np.max(error),
            'First pair flat km': flat[0] / 1000})
projection_accuracy = pd.DataFrame(projection_rows)
display(projection_accuracy)
print('First pair:', gdf.iloc[pi[0]].COCNUM, gdf.iloc[pj[0]].COCNUM, 'Earth distance km:', earth[0] / 1000)

# Repair polygon shapes on a separate copy and check how far their center points move.
# A tiny movement means this repair has little effect on the centroid-distance analysis.
invalid = ~projected.geometry.is_valid
repaired = projected.geometry.make_valid()
displacement = projected.geometry.centroid.distance(repaired.centroid)
geometry_evidence = pd.DataFrame({'COCNUM': gdf.COCNUM,
        'Originally valid': ~invalid,
        'Centroid displacement m': displacement})
display(geometry_evidence[invalid])
print('Invalid polygons:', invalid.sum(), 'Maximum displacement m:', displacement.max())


## Step 22 — Export geography for ArcGIS checking

**1. What this step does**

- Provide the same study geometry and representative points to GIS tools.

1. Calculate centroids in the analysis projection
2. Transform points to longitude/latitude for display
3. Export the coordinate table, shapefile and GeoPackage


In [ ]:
# Step 22: Export geography for ArcGIS checking
# Export the 141 study polygons, homelessness counts and calculated center points for GIS
# checking. Distances use the regional projection in metres; exported longitude and latitude
# help place the same points in ArcGIS.

import os
import pandas as pd
import geopandas as gpd
if 'gdf' not in globals():
    raise NameError('Run Cell 22 first to create gdf.')

# Find each polygon’s center in the analysis projection, then convert it to longitude/latitude.
# Distances use metre coordinates; longitude/latitude coordinates help display points in GIS.
gdf_proj = gdf.to_crs(ANALYSIS_CRS).copy()
analysis_centroids = gpd.GeoSeries(gdf_proj.geometry.centroid, crs=ANALYSIS_CRS)
centroids_wgs84 = analysis_centroids.to_crs(epsg=4326)
gdf_wgs84 = gdf.to_crs(epsg=4326).copy()
gdf_wgs84['centroid_lon'] = centroids_wgs84.x.to_numpy()
gdf_wgs84['centroid_lat'] = centroids_wgs84.y.to_numpy()
export_cols = ['COCNUM', 'COCNAME', 'homeless', 'centroid_lon', 'centroid_lat']

# Compare placeholder coordinates only if those optional fields are available.
if 'x' in gdf_wgs84.columns and 'y' in gdf_wgs84.columns:
    gdf_wgs84['csv_x'] = pd.to_numeric(gdf_wgs84['x'], errors='coerce')
    gdf_wgs84['csv_y'] = pd.to_numeric(gdf_wgs84['y'], errors='coerce')
    export_cols += ['csv_x', 'csv_y']
    csv_points = gpd.GeoSeries(gpd.points_from_xy(gdf_wgs84['csv_x'],
            gdf_wgs84['csv_y']),
        crs='EPSG:4326').to_crs(ANALYSIS_CRS)
    true_centroids = analysis_centroids
    gdf_wgs84['dist_csv_to_centroid_m'] = csv_points.distance(true_centroids)
    export_cols.append('dist_csv_to_centroid_m')
os.makedirs('outputs/arcgis_export', exist_ok=True)

# Save the coordinate table and GIS boundary formats for separate ArcGIS verification.
csv_out = 'outputs/arcgis_export/coc_spatial_verification.csv'
gdf_wgs84[export_cols].to_csv(csv_out, index=False)
print(f'Saved CSV: {csv_out}')
print('Rows:', len(gdf_wgs84))
shp_dir = 'outputs/arcgis_export'
os.makedirs(shp_dir, exist_ok=True)
shp_out = os.path.join(shp_dir, 'coc_boundaries_verification.shp')
gdf_export = gdf_wgs84.copy()
rename_map = {'centroid_lon': 'ctr_lon', 'centroid_lat': 'ctr_lat', 'dist_csv_to_centroid_m': 'dist_m'}
gdf_export = gdf_export.rename(columns={k: v for k, v in rename_map.items() if k in gdf_export.columns})
gdf_export.to_file(shp_out)
print(f'Saved shapefile: {shp_out}')
gpkg_out = os.path.join(shp_dir, 'coc_boundaries_verification.gpkg')
gdf_export.to_file(gpkg_out, layer='coc_boundaries', driver='GPKG')
print(f'Saved GeoPackage: {gpkg_out}')
display(gdf_wgs84[export_cols].head(10))
if 'dist_csv_to_centroid_m' in gdf_wgs84.columns:
    print('\nDistance CSV x/y -> analysis centroid (projected metres):')
    print(gdf_wgs84['dist_csv_to_centroid_m'].describe())


## Step 23 — Test OLS residual clustering

**1. What this step does**

- Calculate Moran's I for the primary model errors.

1. Use observed-minus-predicted residuals from the existing fit
2. Join geometry and connect five nearest areas
3. Row-standardize weights and run 9,999 residual permutations with the fixed seed


In [ ]:
# Step 23: Test OLS residual clustering
# Calculate each OLS error: recorded count minus prediction. Ask whether nearby areas have
# similar errors using Moran’s I and five nearest neighbors. Use 9,999 rearrangements to compare
# the observed pattern with a random-location reference.

from libpysal.weights import KNN, lag_spatial
from esda.moran import Moran
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Set 9,999 random rearrangements and a seed so repeated runs use the same random sequence.
MORAN_PERMUTATIONS = 9999
MORAN_SEED = 5536
print('Model 1 predictors used for residual Moran:')
for col in MODEL1_PREDICTORS:
    print(f'  - {col}')

# Calculate each error as observed count minus prediction, then match it to the area’s polygon.
# Example: observed 100, predicted 80 -> residual +20, meaning underprediction.
resid_df = pd.DataFrame({'COCNUM': merged_df.loc[y_model.index, 'COCNUM'].values, 'residual': y_model.values - y_pred})
gdf_res = gdf.merge(resid_df, on='COCNUM', how='inner')
gdf_res = gdf_res.dropna(subset=['residual']).to_crs(ANALYSIS_CRS)

# Give each area five nearest neighbors and weights that sum to 1.
# Here each of the five neighbors has weight 1/5. Moran’s I compares neighboring residuals.
w = KNN.from_dataframe(gdf_res, k=5)
w.transform = 'r'
res_vals = gdf_res['residual'].values
np.random.seed(MORAN_SEED)
mi_res = Moran(res_vals, w, permutations=MORAN_PERMUTATIONS)
print('=' * 60)
print("Moran's I on Model 1 residuals (KNN k=5)")
print('=' * 60)
print(f'CoCs analyzed:          {len(gdf_res)}')
print(f"Moran's I (residuals):  {mi_res.I:.6f}")
print(f'Expected I (null):      {mi_res.EI:.6f}')
print(f'p-value (simulation):   {mi_res.p_sim:.4f}')
print(f'Permutations:            {MORAN_PERMUTATIONS}')
print(f'Random seed:             {MORAN_SEED}')
print('=' * 60)

# Plot each standardized residual against its weighted neighbor residual.
res_std = (res_vals - res_vals.mean()) / res_vals.std()
res_lag = lag_spatial(w, res_std)
plt.figure(figsize=(7, 6))
plt.scatter(res_std, res_lag, alpha=0.7, edgecolor='k', linewidth=0.3)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('Standardized residual (this CoC)')
plt.ylabel('Spatial lag of residuals (neighbors)')
plt.title(f'Residual Moran Scatter (I={mi_res.I:.4f}, p={mi_res.p_sim:.4f})')
plt.tight_layout()
plt.show()


## Step 24 — Show the actual Moran comparison

**1. What this step does**

- Explain the observed statistic and randomization using real data.

1. Display DC's residual neighbors and weights
2. Reconstruct Moran's I using centered errors
3. Count permutation exceedances and draw their distribution


In [ ]:
# Step 24: Show the actual Moran comparison
# Show DC-500’s actual five residual neighbors and rebuild the Moran statistic from the data.
# Count the relevant simulated results to reconstruct the reported p-value. This makes the
# calculation behind the spatial test visible.

# Show DC-500’s actual residual neighbors and weights.
focal = int(np.flatnonzero(gdf_res.COCNUM.eq('DC-500'))[0])
neighbors = w.neighbors[focal]
neighbor_table = gdf_res.iloc[neighbors][['COCNUM', 'residual']].copy()
neighbor_table['Weight'] = w.weights[focal]
display(neighbor_table)
print('DC residual:', gdf_res.iloc[focal].residual)

# Recalculate Moran’s I from centered residuals and neighbor averages.
# For p_sim, use the smaller tail count and add 1 to both count and permutation total.
# Example: 8 tail events in 9,999 shuffles give (8+1)/(9,999+1) = .0009.
z = res_vals - res_vals.mean()
manual_i = len(z) / w.s0 * np.dot(z, lag_spatial(w, z)) / np.dot(z, z)

# Count simulations at least as large as observed; use PySAL’s smaller-tail rule and plus-one correction.
upper = int(np.count_nonzero(mi_res.sim >= mi_res.I))
tail = min(upper, MORAN_PERMUTATIONS - upper)
display(pd.DataFrame({'Quantity': ['Calculated I',
                'Expected I',
                'Upper-tail exceedances',
                'Smaller-tail count',
                'p_sim'],
            'Value': [manual_i,
                mi_res.EI,
                upper,
                tail,
                (tail + 1) / (MORAN_PERMUTATIONS + 1)]}))
np.testing.assert_allclose(manual_i, mi_res.I)
np.testing.assert_allclose((tail + 1) / (MORAN_PERMUTATIONS + 1), mi_res.p_sim)

# Compare the observed statistic with the actual rearrangement distribution.
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(mi_res.sim, bins=40, color='steelblue')
ax.axvline(mi_res.I, color='firebrick', label='Observed I')
ax.set(xlabel='Moran I after rearrangement', ylabel='Permutations', title='Actual OLS permutation evidence')
ax.legend()
plt.show()


## Step 25 — Test residuals using distance bands

**1. What this step does**

- Check clustering under alternative definitions of nearby.

1. Build inverse-distance weights for 50–500 km bands
2. Exclude CoCs with no neighbor within each band
3. Recalculate Moran's I and p_sim with the documented seeds


In [ ]:
# Step 25: Test residuals using distance bands
# Repeat the OLS residual-clustering check using nearby areas within 50, 100, ... 500 km. Areas
# without a neighbor inside a band are excluded for that band. Each result therefore reflects
# both a distance rule and its included sample.

from libpysal.weights import DistanceBand
from esda.moran import Moran
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
gdf_proj = gdf_res.dropna(subset=['residual']).to_crs(ANALYSIS_CRS).copy()
target_col = 'residual'
rows = []

# Repeat Moran’s I at 50, 100, ... 500 km. Within each band, nearer neighbors get more weight.
for km in range(50, 501, 50):
    threshold_m = km * 1000
    w = DistanceBand.from_dataframe(gdf_proj, threshold=threshold_m, binary=False, silence_warnings=True)

    # Leave out an area for this test if it has no neighbor within the selected distance.
    # Example: a nearest neighbor 70 km away cannot be used in the 50 km test.
    islands = set(w.islands)
    if len(islands) >= len(gdf_proj) - 1:
        rows.append({'Distance (km)': km, "Moran's I (residual)": np.nan, 'p-value': np.nan, 'CoCs used': 0})
        continue
    keep_idx = [i for i in range(len(gdf_proj)) if i not in islands]
    gdf_conn = gdf_proj.iloc[keep_idx].copy()
    w_conn = DistanceBand.from_dataframe(gdf_conn, threshold=threshold_m, binary=False, silence_warnings=True)
    w_conn.transform = 'r'
    np.random.seed(MORAN_SEED + km)
    mi = Moran(gdf_conn[target_col].values, w_conn, permutations=MORAN_PERMUTATIONS)
    rows.append({'Distance (km)': km, "Moran's I (residual)": mi.I, 'p-value': mi.p_sim, 'CoCs used': len(gdf_conn)})

# Summarize and plot the band-specific residual clustering results.
resid_dist_df = pd.DataFrame(rows)
display(resid_dist_df)
valid = resid_dist_df.dropna(subset=["Moran's I (residual)"])
plt.figure(figsize=(7, 4))
plt.plot(valid['Distance (km)'], valid["Moran's I (residual)"], marker='o')
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('Distance (km)')
plt.ylabel("Moran's I (residual)")
plt.title(f"Residual Moran's I vs Distance-band threshold ({MORAN_PERMUTATIONS:,} permutations)")
plt.tight_layout()
plt.show()


## Step 26 — Make distance-band exclusions explicit

**1. What this step does**

- Display sample changes alongside each band's result.

1. Copy the band table
2. Subtract included CoCs from the full sample
3. Mark reported p-values below .05


In [ ]:
# Step 26: Make distance-band exclusions explicit
# Add how many CoCs each distance-band test excluded and whether its p-value meets .05. This
# prevents treating differences between tests as distance effects alone when their samples also
# differ.

distance_evidence = resid_dist_df.copy()
distance_evidence['CoCs excluded'] = len(gdf_res) - distance_evidence['CoCs used']
distance_evidence['p < .05'] = distance_evidence['p-value'] < 0.05
display(distance_evidence)
print('All tested bands show supported positive clustering:',
    bool(((distance_evidence["Moran's I (residual)"] > 0) & distance_evidence['p < .05']).all()))


### Interpreting distance-band changes

- Each band changes the definition of nearby and may change the sample by excluding isolated CoCs.
- Read sample size together with Moran's I.
- Differences are not pure distance effects.
- The main five-neighbor test retains all 141 CoCs.

## Step 27 — Prepare the data for GWR: how it differs from OLS

**1. What we have done so far: OLS**

- OLS means Ordinary Least Squares. It fits one shared equation across all 141 CoCs.
- Each predictor has one coefficient (weight) that is used for every CoC.
- Each area still gets its own prediction because its population, crime and other input values differ.

**2. What we do next: GWR**

- GWR means Geographically Weighted Regression. It fits a local equation around each CoC.
- Nearby observations receive greater weight; sufficiently distant ones receive zero weight with our bi-square kernel.
- Each CoC therefore gets its own fitted coefficients. A predictor’s estimated relationship can be stronger, weaker or point in a different direction in different places.
- These local estimates borrow information from neighboring areas; they are not fitted from a single CoC alone.

| Question | OLS | GWR |
|---|---|---|
| How many equations? | One shared equation | A local equation around each CoC |
| Which observations contribute? | All study observations have equal fitting weight | Weights depend on distance from the focal CoC |
| Does a predictor have the same coefficient everywhere? | Yes | No: its coefficient can vary by location |
| Does every CoC get the same predicted count? | No: its own input values determine its prediction | No: its inputs and local coefficients determine its prediction |
| What does it help us see? | A global reference relationship across the study | How fitted relationships differ geographically |

**3. A simple example**

- Imagine examining the relationship between Crime Index and homelessness.
- OLS gives Crime Index one shared coefficient for the whole study.
- GWR can estimate a positive coefficient around one CoC and a negative coefficient around another, because the weighted local data differ.
- We later check statistical support before calling an opposite sign a significant reversal. These associations do not establish causation.

**4. Why compare them in this project?**

- Our question is whether the relationships in the global equation also describe local areas consistently.
- Keep the same four selected predictors, homelessness outcome, 141 CoCs and scaling so the global and local coefficients can be compared.
- Compare direction, statistical support, magnitude and geographic variability in later steps.
- A higher GWR R² on fitted data does not by itself prove better prediction for new areas.

**5. What the code in Step 27 does**

1. Check that the earlier model and geographic data are available.
2. Gather the CoC codes, four selected predictor values and recorded homelessness counts.
3. Attach each CoC’s boundary polygon using its code.
4. Verify that the GWR table contains the same number of observations as the primary OLS model.

- This step prepares the table. The later coordinate, bandwidth and fitting steps calculate the GWR models.
- Moran’s I is a separate check: it examines whether nearby areas have similar model errors. It does not fit the local equations.


In [ ]:
# Step 27: Build the GWR study table
# Prepare GWR using the same four predictors and outcome rows as the primary OLS model. Join
# geographic polygons by CoC code and check the sample count so later global/local comparisons
# use matching areas.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error
from mgwr.gwr import GWR
from mgwr.sel_bw import Sel_BW
from libpysal.weights import KNN
from esda.moran import Moran

# Require the completed primary model and geographic data.
required = ['MODEL1_PREDICTORS', 'X_model', 'y_model', 'y_pred', 'model', 'r2', 'rmse', 'mi_res', 'merged_df', 'gdf']
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f'Run earlier cells first — missing: {missing}')
print('GWR predictors (same as Model 1):')
for col in MODEL1_PREDICTORS:
    print(f'  - {col}')

# Build GWR from the same selected predictor rows and outcome as OLS.
gwr_df = pd.concat([merged_df.loc[y_model.index, 'COCNUM'], X_model, y_model.rename('y')], axis=1).dropna()

# Attach geometry by CoC code and check sample size.
gdf_geom = gdf[['COCNUM', 'geometry']].copy()
gwr_gdf = gdf_geom.merge(gwr_df, on='COCNUM', how='inner')
assert len(gwr_gdf) == len(y_model), f'Row mismatch: GWR={len(gwr_gdf)}, Model 1={len(y_model)}'
print(f'\nGWR sample: {len(gwr_gdf)} CoCs | predictors: {len(MODEL1_PREDICTORS)}')
display(gwr_gdf[['COCNUM', 'y'] + MODEL1_PREDICTORS].head())


## Step 28 — Calculate GWR representative coordinates

**1. What this step does**

- Convert each study polygon into a projected centroid point.

**2. Why we do it**

- GWR needs distances between representative locations.

**3. Process, in order**

1. Project polygons to regional AEQD
2. Calculate centroid x and y in metres
3. Pair the coordinates in table order

**4. How to read the output**

- Centroids are geometric representatives, not necessarily population centers or service locations.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 28: Calculate GWR representative coordinates
# Represent each CoC by its polygon centroid in the regional metre-based projection. Store the
# x/y coordinates together in row order so GWR can calculate distances between the correct
# observations.

# Project the polygons into metres and use each polygon’s centroid as its representative point.
gwr_gdf = gwr_gdf.to_crs(ANALYSIS_CRS).copy()
gwr_gdf['coord_x'] = gwr_gdf.geometry.centroid.x
gwr_gdf['coord_y'] = gwr_gdf.geometry.centroid.y
coords = list(zip(gwr_gdf['coord_x'], gwr_gdf['coord_y']))


## Step 29 — Prepare GWR numerical inputs

**1. What this step does**

- Create the outcome and predictor arrays required by GWR.

**2. Why we do it**

- The fitting library expects numerical matrices in the same row order.

**3. Process, in order**

1. Reshape raw y as one column
2. Scale the four predictors to 0–1
3. Preserve the order of predictors and areas

**4. How to read the output**

- The later check compares both scalers' minima and maxima. Different scales would make coefficient comparisons misleading.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 29: Prepare GWR numerical inputs
# Give GWR a one-column array of homelessness counts and a matrix of scaled predictors. Scale
# each predictor from 0 to 1 while keeping counts in people. Preserve the same row order as the
# coordinates.

# Put counts in a one-column array, keeping the same row order as locations and predictors.
y_gwr = gwr_gdf['y'].values.reshape((-1, 1))

# Put each predictor on a 0–1 scale: (value - minimum) / (maximum - minimum).
# Example: minimum 10, maximum 30, value 20 -> scaled value .5.
scaler_gwr = MinMaxScaler()

# Fit the scaling on these same primary-model observations so local/global coefficients are comparable.
X_gwr = scaler_gwr.fit_transform(gwr_gdf[MODEL1_PREDICTORS])


### How GWR chooses its neighborhood size

- GWR fits a separate, weighted regression for each of the 141 CoCs.
- Nearby CoCs receive more weight.
- The **adaptive bandwidth** tells GWR how many nearby observations to use at each location; the distance covered by that neighborhood can differ between places.

- The bandwidth is **calculated each time**, not entered as 78 or 53.
- The `Sel_BW(..., fixed=False)` step tries candidate neighborhood sizes and selects the one with the lowest corrected Akaike Information Criterion (**AICc**).
- This weighs fit against model complexity.
- The following `GWR(..., bw)` step uses that selected value.

- An earlier version of the analysis reported **78 neighbors**.
- The saved output for the **current predictor set and data** reports **53 neighbors**.
- A different model specification or set of locations can change the best bandwidth.
- We have not reproduced the older run under controlled conditions, so we cannot assign the difference to one particular variable removal.
- For this notebook, use the number printed by the current bandwidth-selection cell and rerun all downstream GWR results if the model or spatial coordinates change.

- With 53 neighbors, local estimates use a smaller neighborhood than with 78, allowing more local variation; this can also make individual local estimates less stable.
- AICc selection by itself does not prove that 53 is the best choice for every scientific purpose.

## Step 30 — Select the adaptive GWR bandwidth

**1. What this step does**

- Choose the neighborhood size using the current inputs.

**2. Why we do it**

- Neighborhood size controls how local the fitted relationships are.

**3. Process, in order**

1. Try candidate neighborhood sizes
2. Evaluate AICc
3. Retain the size with the best criterion among those evaluated

**4. How to read the output**

- The selected value is 53. Earlier 78 belongs to an older run; this notebook does not establish a single cause for that historical change.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.

### Reading the GWR parameter summary: X0–X4

- Step 30 selects the bandwidth. Step 31 fits GWR and prints the parameter summary described here.
- X0–X4 are short labels the GWR library gives to the parameters, in the order supplied to the model.

| Label | What it represents |
|---|---|
| X0 | Intercept: the equation’s starting number |
| X1 | Hispanic Pacific Islander population |
| X2 | Civilian veteran population aged 18+ |
| X3 | Total Crime Index |
| X4 | Tapestry Segment 12D population aged 18+ |

- There are four predictors plus the intercept, so five rows appear.
- Predictor coefficients use the same min-max-scaled inputs as the primary OLS model.
- GWR estimates each parameter at every one of the 141 CoCs. This table summarizes those 141 local coefficients for each parameter.

| Summary column | Meaning |
|---|---|
| Mean | Average of the 141 local coefficients |
| STD | Standard deviation: how spread out those coefficients are |
| Min | Smallest local coefficient |
| Median | Middle local coefficient after sorting |
| Max | Largest local coefficient |

- **Example:** X3 is Total Crime Index. Its local coefficients range from approximately **−6,614.879 to +3,489.638** in the current run. The fitted relationship is negative in some locations and positive in others.
- These are equation coefficients, not raw crime counts or predicted homelessness counts. They describe changes in estimated homelessness per unit of the scaled predictor, holding other inputs fixed.
- This summary alone does not tell us which local coefficients are statistically significant. Later steps compare their t-values with the adjusted cutoff before counting significant reversals.


In [ ]:
# Step 30: Select the adaptive GWR bandwidth
# Choose how many observations contribute to each local GWR neighborhood. Compare candidate
# bandwidths using AICc, which balances fit and complexity. The current selection is 53
# observations including the focal area, with distance-dependent weights.

# Try adaptive neighborhood sizes and select the one with the lowest AICc.
# The current selected bandwidth is 53 observations, including the focal observation; weights vary.
bw_selector = Sel_BW(coords, y_gwr, X_gwr, fixed=False)
bw = bw_selector.search(criterion='AICc')
print(f'\nOptimal adaptive bandwidth: {bw} nearest neighbors')


## Step 31 — Fit the local GWR models

**1. What this step does**

- Estimate coefficients for each CoC using nearby observations.

**2. Why we do it**

- One global coefficient may not describe every place equally well.

**3. Process, in order**

1. Construct GWR with selected bandwidth and inputs
2. Weight nearer observations more strongly with the default bisquare kernel
3. Fit and store local coefficients, uncertainty and performance

**4. How to read the output**

- Each predictor has 141 local coefficients. A local fitted association is not a causal intervention effect.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 31: Fit the local GWR models
# Fit a local regression around each CoC. Nearby observations receive more weight under the
# adaptive bi-square kernel. The fitted result contains a different set of coefficients at each
# area, plus predictions and uncertainty.

# Fit one local equation per area using its adaptive bi-square distance weights.
gwr_model = GWR(coords, y_gwr, X_gwr, bw)
gwr_results = gwr_model.fit()
print(gwr_results.summary())


## Real GWR neighborhood and threshold derivation — Corrections 11–13, 24, 27

- GWR uses an adaptive nearest-observation bandwidth selected by AICc.
- The focal CoC participates.
- Bisquare weights taper near the neighborhood boundary; 53 does not mean 53 equally weighted other areas.
- Retrieve DC's actual weights and reconstruct its weighted regression.

- The adjusted .05 alpha is .05×k/ENP, where k includes intercept and ENP is effective parameters.
- The installed library derives critical t with n−1 degrees of freedom and adjusted alpha/2 in each tail.
- Compute this instead of hard-coding 2.6303.
- Passing it supports a fitted local association under the model assumptions, not causation or robustness to every specification.
- Matching samples and scaling makes OLS/local comparisons meaningful.

## Step 32 — Reconstruct one neighborhood and its significance threshold

**1. What this step does**

- Show where a local estimate and adjusted test come from.

**2. Why we do it**

- The reader should be able to follow the calculation rather than accept a hard-coded threshold.

**3. Process, in order**

1. Retrieve DC's actual distance weights
2. Reconstruct its weighted least-squares coefficients
3. Calculate adjusted alpha and critical t; check shared scaling and sample codes

**4. How to read the output**

- The reconstructed coefficients match GWR, and critical |t|≈2.6303. The focal area participates; bandwidth 53 is not 53 equally weighted other CoCs.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 32: Reconstruct one neighborhood and its significance threshold
# Rebuild DC-500’s local equation using its actual neighborhood weights and compare it with the
# library result. Calculate the adjusted local testing cutoff, then verify that GWR and OLS use
# the same areas and predictor scaling.

from scipy.stats import t as student_t

# Retrieve the actual weights used around DC-500 and rebuild its local regression.
# Larger weights give nearby observations more influence; zero weights exclude distant ones.
focal_gwr = int(np.flatnonzero(gwr_gdf.COCNUM.eq('DC-500'))[0])
wi = gwr_results.model._build_wi(focal_gwr, bw)
xy = np.asarray(coords)
neighborhood = pd.DataFrame({'COCNUM': gwr_gdf.COCNUM.values,
        'Distance km': np.linalg.norm(xy - xy[focal_gwr],
            axis=1) / 1000,
        'Weight': wi})
display(neighborhood[neighborhood.Weight > 0].sort_values('Distance km'))

# Solve the actual weighted least-squares equation. Multiplying rows by square-root weights
# makes their squared errors contribute in proportion to the GWR neighborhood weights.
weighted_beta = np.linalg.lstsq(gwr_results.X * np.sqrt(wi[:,
            None]),
    y_gwr * np.sqrt(wi[:,
            None]),
    rcond=None)[0].ravel()
display(pd.DataFrame({'Parameter': ['Intercept'] + MODEL1_PREDICTORS,
            'Reconstructed coefficient': weighted_beta,
            'GWR coefficient': gwr_results.params[focal_gwr]}))
np.testing.assert_allclose(weighted_beta, gwr_results.params[focal_gwr], rtol=1e-06, atol=1e-05)

# Calculate the local testing cutoff from mgwr’s adjusted alpha and the study sample size.
# This produces the cutoff used later to decide whether a local |t| is large enough.
adjusted_alpha = float(gwr_results.adj_alpha[1])
critical = float(student_t.ppf(1 - adjusted_alpha / 2, gwr_results.n - 1))
display(pd.DataFrame({'Quantity': ['Bandwidth',
                'n',
                'k including intercept',
                'ENP',
                'Adjusted alpha',
                'Critical absolute t'],
            'Value': [bw,
                gwr_results.n,
                gwr_results.k,
                gwr_results.ENP,
                adjusted_alpha,
                critical]}))
np.testing.assert_allclose(critical, gwr_results.critical_tval(None))
np.testing.assert_allclose(scaler_gwr.data_min_, scaler.data_min_)
np.testing.assert_allclose(scaler_gwr.data_max_, scaler.data_max_)

# Verify that local/global comparisons use the same CoCs and predictor scaling.
assert set(gwr_gdf.COCNUM) == set(prediction_evidence.COCNUM)


## Step 33 — Store local predictions and coefficients

**1. What this step does**

- Attach the fitted GWR outputs to their CoC rows.

**2. Why we do it**

- Later tables and maps need correctly labeled local results.

**3. Process, in order**

1. Save fitted counts, residuals and local R²
2. Name intercept and predictor parameters
3. Create a separate coefficient column for each predictor

**4. How to read the output**

- Read each result beside its CoC code. This step stores an existing fit; it does not refit or classify significance.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 33: Store local predictions and coefficients
# Attach every GWR prediction, residual, local R² and coefficient to its own CoC row. Keeping
# names and values together allows later maps and tables to show the correct result at each
# location.

# Store each local prediction, error and local fit score alongside its area.
# Residual = observed count - local prediction.
gwr_gdf['gwr_pred'] = gwr_results.predy.flatten()
gwr_gdf['gwr_resid'] = gwr_results.resid_response.flatten()
gwr_gdf['gwr_local_R2'] = gwr_results.localR2.flatten()

# Give each local coefficient a column named for its predictor.
# Parameter column 0 is the intercept; predictor coefficients start at column 1.
param_names = ['Intercept'] + MODEL1_PREDICTORS
for i, name in enumerate(param_names):
    gwr_gdf[f'gwr_coef_{name}'] = gwr_results.params[:, i]


## Step 34 — Optional housing-coefficient export

**1. What this step does**

- Export a Housing Affordability local coefficient only if that predictor was selected.

**2. Why we do it**

- An unselected predictor must not be presented as an estimated primary-model coefficient.

**3. Process, in order**

1. Check whether the coefficient column exists
2. Export it if present
3. Otherwise print a clear skip message

**4. How to read the output**

- This run skips the export because Housing Affordability is absent from the primary model. Its descriptive Figure 3 is a different use of the raw variable.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 34: Optional housing-coefficient export
# Export a housing-affordability local coefficient only if housing was selected in the primary
# model. The current four-predictor model excludes housing, so this cell skips the export rather
# than inventing a coefficient.

import os
hai_raw = '2024 Housing Affordability Index'
hai_col = f'gwr_coef_{hai_raw}'
os.makedirs('outputs/arcgis_export', exist_ok=True)

# Export only if housing was selected; this primary run correctly prints a skip message.
if hai_col in gwr_gdf.columns:
    out = gwr_gdf[['COCNUM', 'y', hai_col]].copy()
    out = out.rename(columns={'y': 'homeless', hai_col: 'gwr_hai'})
    out.to_csv('outputs/arcgis_export/gwr_housing_affordability.csv', index=False)
    print('Saved: outputs/arcgis_export/gwr_housing_affordability.csv')
    print(out.head())
else:
    print(f"Skipped HAI export — '{hai_raw}' not in MODEL1_PREDICTORS.")


## Step 35 — Export veteran local coefficients

**1. What this step does**

- Save keyed veteran population and local fitted relationships for GIS use.

**2. Why we do it**

- External coefficient maps need the correct values per CoC.

**3. Process, in order**

1. Check that the veteran coefficient exists
2. Select code, outcome, population and coefficient
3. Rename fields and save the CSV

**4. How to read the output**

- The export runs because veterans are a selected predictor. A coefficient must not be confused with the veteran population itself.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 35: Export veteran local coefficients
# Export veteran population and its GWR coefficient beside each CoC code for GIS use. The
# population count describes the area; the coefficient describes its fitted local relationship.
# They are separate quantities.

import os
vet_raw = '2022 Civilian Pop 18+: Veteran (ACS 5-Yr)'
vet_coef = f'gwr_coef_{vet_raw}'
os.makedirs('outputs/arcgis_export', exist_ok=True)

# Save raw veteran population separately from its fitted local coefficient.
if vet_coef in gwr_gdf.columns:
    cols = ['COCNUM', 'y', vet_coef]
    if vet_raw in gwr_gdf.columns:
        cols = ['COCNUM', 'y', vet_raw, vet_coef]
    out = gwr_gdf[cols].copy()
    rename = {'y': 'homeless', vet_coef: 'gwr_vet'}
    if vet_raw in out.columns:
        rename[vet_raw] = 'vet_pop'
    out = out.rename(columns=rename)
    out.to_csv('outputs/arcgis_export/gwr_veteran.csv', index=False)
    print('Saved: outputs/arcgis_export/gwr_veteran.csv')
    print(out.head())
else:
    print(f"Skipped Veteran export — '{vet_raw}' not in MODEL1_PREDICTORS.")


## Step 36 — Compare OLS and GWR fit and coefficients

**1. What this step does**

- Bring global and local fitted outputs together.

**2. Why we do it**

- Fit and geographic variation are different questions.

**3. Process, in order**

1. Calculate GWR R² and RMSE from its actual predictions
2. Print both models' fit and GWR AICc
3. Summarize the local coefficient distribution beside OLS

**4. How to read the output**

- GWR R²≈.9802 and RMSE≈807.84 describe closer in-sample fit. This alone does not establish better forecasting.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 36: Compare OLS and GWR fit and coefficients
# Compare OLS and GWR estimates with the recorded counts using R² and RMSE. Also compare each
# single OLS coefficient with the range and median of its GWR coefficients. These describe
# fitted data rather than unseen-area accuracy.

# Compare fitted counts with observations to calculate GWR in-sample performance.
gwr_r2 = r2_score(gwr_gdf['y'], gwr_gdf['gwr_pred'])
gwr_rmse = np.sqrt(mean_squared_error(gwr_gdf['y'], gwr_gdf['gwr_pred']))
print('\n' + '=' * 60)
print('OLS (Model 1) vs GWR')
print('=' * 60)
print(f"{'Metric':<22} {'OLS':<18} {'GWR':<18}")
print(f"{'R-squared':<22} {r2:.4f}{'':<10} {gwr_r2:.4f}")
print(f"{'RMSE':<22} {rmse:,.2f}{'':<6} {gwr_rmse:,.2f}")
print(f"{'AICc':<22} {'—':<18} {gwr_results.aicc:.2f}")
print(f"{'Bandwidth (k)':<22} {'—':<18} {bw}")
print('=' * 60)

# Compare each single OLS coefficient with the mean, median, minimum and maximum of its local values.
coef_summary = pd.DataFrame({'variable': MODEL1_PREDICTORS,
        'OLS_coef': model.coef_,
        'GWR_mean': [gwr_gdf[f'gwr_coef_{v}'].mean() for v in MODEL1_PREDICTORS],
        'GWR_min': [gwr_gdf[f'gwr_coef_{v}'].min() for v in MODEL1_PREDICTORS],
        'GWR_median': [gwr_gdf[f'gwr_coef_{v}'].median() for v in MODEL1_PREDICTORS],
        'GWR_max': [gwr_gdf[f'gwr_coef_{v}'].max() for v in MODEL1_PREDICTORS]})
print('\nCoefficient ranges (OLS = global; GWR = local across CoCs):')
display(coef_summary)


## Step 37 — Plot local fit and coefficient values

**1. What this step does**

- Show how fitted GWR values vary across the study.

**2. Why we do it**

- Maps reveal location-specific patterns that a single summary hides.

**3. Process, in order**

1. Map local R²
2. Draw one coefficient map per selected predictor
3. Use 5th–95th percentile color limits for coefficient display

**4. How to read the output**

- These are continuous coefficient maps, not significance-category maps. Colors at the scale limits may include more extreme values.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 37: Plot local fit and coefficient values
# Draw maps of local fit and coefficient values so geographic differences are visible.
# Coefficient map colors describe size and direction; these maps do not classify statistical
# significance. Later category maps perform that task.

# Map local R² to show fitted geographic variation.
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
gwr_gdf.plot(column='gwr_local_R2', cmap='YlOrRd', legend=True, ax=ax, edgecolor='black', linewidth=0.2)
ax.set_title('GWR Local R² by CoC')
ax.axis('off')
plt.tight_layout()
plt.show()

# Color the maps by coefficient value using the 5th–95th percentiles as display limits.
# These colors show coefficient size; they do not identify statistical significance.
for pred in MODEL1_PREDICTORS:
    coef_col = f'gwr_coef_{pred}'
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    gwr_gdf.plot(column=coef_col,
        cmap='coolwarm',
        legend=True,
        ax=ax,
        edgecolor='black',
        linewidth=0.2,
        vmin=gwr_gdf[coef_col].quantile(0.05),
        vmax=gwr_gdf[coef_col].quantile(0.95))
    ax.set_title(f'Local GWR coefficient: {pred}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()


## Step 38 — Check GWR residual clustering

**1. What this step does**

- Repeat the main residual test after fitting GWR.

**2. Why we do it**

- A closer fit may still leave geographically similar errors.

**3. Process, in order**

1. Create the same five-neighbor structure on GWR rows
2. Run 9,999 permutations with the GWR seed
3. Compare observed I and p_sim with OLS

**4. How to read the output**

- GWR I≈.0139 and p_sim≈.3258 do not detect clustering in this test. Nonsignificance does not prove all spatial problems are gone.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 38: Check GWR residual clustering
# Test GWR errors with the same five-neighbor Moran method used for OLS. This checks whether
# nearby areas still have similar errors after local fitting. Compare both results without
# equating nonsignificance with proof of no spatial pattern.

# Use the same five-neighbor residual test with the documented GWR seed.
w_gwr = KNN.from_dataframe(gwr_gdf, k=5)
w_gwr.transform = 'r'
np.random.seed(MORAN_SEED + 100000)
mi_gwr_resid = Moran(gwr_gdf['gwr_resid'].values, w_gwr, permutations=MORAN_PERMUTATIONS)
print('\n' + '=' * 60)
print("Residual Moran's I: OLS vs GWR (KNN k=5)")
print('=' * 60)
print(f"{'Model':<20} {'Moran I':<15} {'p-value':<15}")
print(f"{'OLS residuals':<20} {mi_res.I:.6f}      {mi_res.p_sim:.4f}")
print(f"{'GWR residuals':<20} {mi_gwr_resid.I:.6f}      {mi_gwr_resid.p_sim:.4f}")
print(f'Permutation settings: {MORAN_PERMUTATIONS:,} | seeds: OLS={MORAN_SEED}, GWR={MORAN_SEED + 100000}')
print('=' * 60)
if mi_gwr_resid.I < mi_res.I:
    print('GWR reduced residual spatial autocorrelation vs OLS.')
else:
    print('GWR did not reduce residual spatial autocorrelation vs OLS.')


## OLS versus GWR: what the comparison shows — Corrections 1, 2, 23

- Both fits use the same sample, outcome, and predictors.
- Higher in-sample R² and lower RMSE describe closer fit on these observations.
- GWR's greater flexibility does not prove held-out accuracy.
- A nonsignificant residual Moran test means that test did not detect clustering, not that all geographic problems disappeared.

## Step 39 — Display the combined fit and residual evidence

**1. What this step does**

- Show both models' fit and remaining clustering in one table.

**2. Why we do it**

- A reader should not substitute R² for a spatial diagnostic.

**3. Process, in order**

1. Create one row per model
2. Place R², RMSE, Moran I and p_sim in separate columns
3. Read the measures together

**4. How to read the output**

- R²/RMSE describe in-sample fit; Moran I/p_sim describe residual arrangement. They answer different questions.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 39: Display the combined fit and residual evidence
# Put OLS and GWR fit scores and residual-clustering results in one comparison table. R²
# describes fitted counts; Moran’s I describes the geographic pattern left in the errors. Both
# are needed to interpret model performance.

comparison_evidence = pd.DataFrame({'Model': ['OLS',
            'GWR'],
        'R²': [r2,
            gwr_r2],
        'RMSE': [rmse,
            gwr_rmse],
        'Residual Moran I': [mi_res.I,
            mi_gwr_resid.I],
        'Permutation p_sim': [mi_res.p_sim,
            mi_gwr_resid.p_sim]})
display(comparison_evidence)


## Phase 5: Interpretation stability + Monte Carlo spatial variability

- After GWR, summarize whether local conclusions are stable:

1. **Direction** — % of CoCs with the same coefficient sign as OLS
2. **Significance** — significant–same / significant–reversal / nonsignificant (mgwr adjusted critical |t|)
3. **Magnitude** — GWR median [IQR] and median % deviation from OLS
4. **Geographic consistency** — Monte Carlo spatial-variability p-values (`n_iters=1000`)

- Uses the same Model 1 predictors, same min-max scaling, and the PIT overall-homeless outcome.

### How to interpret local coefficient reversals

- This section explains the reversal percentages produced by the stability table.

- The global OLS model gives one coefficient for each predictor across all 141 CoCs.
- The GWR model gives one local coefficient for that same predictor at each CoC.
- We compare the sign of each local coefficient with the sign of the OLS coefficient:

- **Same direction:** the local and global coefficients are both positive or both negative.
- **Direction reversal:** the local coefficient has the opposite sign from the global coefficient.
- **Significant reversal:** the direction reversal is also statistically supported because the absolute local t-statistic is larger than the adjusted critical t-value.
- **Nonsignificant local result:** the local estimate is not strong enough to pass the adjusted t threshold. It is not treated as evidence of either direction.

- For example, if the global crime coefficient is positive but a CoC has a negative local crime coefficient, that CoC is a direction reversal.
- It counts as a **significant reversal** only when its local test also passes the adjusted critical threshold.
- This prevents a small, noisy coefficient from being presented as a meaningful finding.

- The notebook records these categories in `outputs/gwr_local_stability_long.csv` and summarizes their percentages in `outputs/interpretation_stability_table.csv`.
- The `Significant_reversal_pct` column is the percentage of all CoCs classified as statistically supported reversals.
- The `Same_sign_pct` column describes directional agreement, even when a local estimate is not statistically significant.

- These results should be reported as spatially varying associations.
- They show where the relationship differs from the global pattern; they do not prove that the predictor causes homelessness.
- A reversal is therefore a useful geographic diagnostic and a reason to investigate local context, not a causal claim.

### Monte Carlo test of geographic variability

- **Question:** Does a predictor's relationship vary across places more than we would expect from a random arrangement?

- A local reversal checks a particular CoC's direction.
- This Monte Carlo test checks the entire coefficient map for a predictor.
- Both answers matter: a relationship can vary greatly in strength across places while keeping the same direction everywhere.

**How the installed mgwr implementation works:**
1. Measure the standard deviation of the fitted local coefficients for each predictor. This measures how spread out its coefficients are across CoCs.
2. Randomly reassign the coordinate pairs among observations, keeping each observation's outcome and predictors together.
3. Select the adaptive bandwidth again with the same selection settings and fit GWR again for that shuffled arrangement.
4. Measure the spread of the new local coefficients.
5. Repeat 1,000 times using seed 5536. For each predictor, count how many shuffled spreads are greater than the observed spread. The library returns that count divided by 1,000.

- **Why we do this:** Local coefficients will usually differ somewhat even in random data.
- The test helps distinguish ordinary variation from an unusually strong geographic pattern.
- A small p-value supports spatial variability in the association; a larger p-value means this test has not established such variability.
- A larger p-value does not prove that every location has the same relationship.

- **Reading the values:** A library p-value of 0.020 means 20 of the 1,000 shuffled coefficient maps had greater spread than the observed map.
- A raw value of 0.000 means none did; it does not mean the probability is literally zero.
- The exported reporting column therefore displays this as '<0.001', while retaining the original numeric library value.
- These are simulation-based results with resolution 0.001.
- Values near 0.05 should be interpreted cautiously because repeating a finite simulation can change them slightly.

- Use these predictor-level results alongside the direction and local-significance table.
- They answer different questions and neither establishes causation.
- The p-values are reported per predictor without an additional adjustment across predictor-level Monte Carlo tests; state this when reporting them.
- The local t threshold uses mgwr's separate adjusted threshold.

- The seed and iteration count are saved with every predictor in `outputs/spatial_variability_pvalues.csv`.
- Results are also joined to the interpretation-stability table.
- The intercept is tested by the library but excluded from the predictor summary.
- This simulation is separate from the 9,999 residual permutations used for Moran's I.

## Explanation of Moran's I using this project

- The following image uses the actual 141 CoCs and the current four-predictor OLS model.
- It is an explanation graphic, not a paper figure.
- Each CoC has one residual: `actual homelessness - model prediction`.
- Moran's I asks whether nearby CoCs tend to have residuals with similar signs and sizes.
- The shuffle comparison keeps the same residual values and the same CoC locations, then randomly reassigns residuals to locations.
- A fixed seed repeats the same shuffle sequence.

![Moran's I using the actual CoC residuals](attachment:morans_real_project_explained.png)

## Step 40 — Calculate interpretation-stability categories

**1. What this step does**

- Compare every local coefficient with the matching global coefficient.

**2. Why we do it**

- Direction, support and magnitude need separate measures.

**3. Process, in order**

1. Compute the adjusted threshold
2. Compare signs and test local |t|
3. Calculate category percentages, medians, quartiles and coefficient deviations; save all local records

**4. How to read the output**

- There are 564 predictor–CoC records. Significant-same, significant-reversal and nonsignificant partition each predictor's 141 CoCs; same-sign agreement overlaps those categories.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 40: Calculate interpretation-stability categories
# Compare each local coefficient with its corresponding OLS coefficient. Classify it as
# significant agreement, significant reversal or nonsignificant. Also summarize sign agreement
# and coefficient differences separately, then save the detailed and summary tables.

import os
import numpy as np
import pandas as pd
required = ['gwr_results', 'gwr_gdf', 'MODEL1_PREDICTORS', 'model', 'ols_final', 'bw']
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f'Run Model 1 + GWR cells first — missing: {missing}')

# Calculate the adjusted local significance threshold from the fitted GWR.
crit_t = float(gwr_results.critical_tval(None))
print(f'Adjusted critical |t|: {crit_t:.4f}')
print(f'GWR adaptive bandwidth: {bw}')
ols_beta = pd.Series(model.coef_, index=MODEL1_PREDICTORS)
rows = []
local_frames = []
n = len(gwr_gdf)

# Compare each predictor’s 141 local estimates with its global OLS reference.
for j, pred in enumerate(MODEL1_PREDICTORS):
    local_b = gwr_results.params[:, j + 1]
    local_se = gwr_results.bse[:, j + 1]

    # Compare coefficient size with uncertainty: t = coefficient / standard error.
    # Example: coefficient -6 and SE 2 give t=-3; the test uses |t|=3.
    local_t = local_b / local_se
    ols_b = float(ols_beta[pred])

    # Compare the signs and test the local t-value calculated above (coefficient / standard error).
    # Example: OLS positive, local -6, SE 2 -> t=-3; |t|=3 exceeds the current 2.6303 cutoff.
    # That qualifies as a significant reversal. Opposite signs with smaller |t| do not.
    same_sign = np.sign(local_b) == np.sign(ols_b)
    if ols_b == 0:
        same_sign = np.isclose(local_b, 0)
    sig = np.abs(local_t) > crit_t

    # Build three non-overlapping groups: supported same direction, supported opposite direction,
    # or below the significance cutoff. Every local record enters exactly one group.
    sig_same = sig & same_sign
    sig_rev = sig & ~same_sign
    nonsig = ~sig

    # Calculate local medians/quartiles and the median absolute percentage difference from OLS.
    # Example: OLS=100, local=150 -> |150-100|/|100| * 100 = 50%.
    med = float(np.median(local_b))
    q1 = float(np.percentile(local_b, 25))
    q3 = float(np.percentile(local_b, 75))
    pct_dev = np.nan
    if ols_b != 0:
        pct_dev = float(np.median(np.abs(local_b - ols_b) / abs(ols_b) * 100))
    rows.append({'Predictor': pred,
            'OLS_beta': ols_b,
            'GWR_median': med,
            'GWR_Q1': q1,
            'GWR_Q3': q3,
            'Same_sign_pct': 100 * same_sign.mean(),
            'Significant_same_pct': 100 * sig_same.mean(),
            'Significant_reversal_pct': 100 * sig_rev.mean(),
            'Nonsignificant_pct': 100 * nonsig.mean(),
            'Median_pct_deviation_from_OLS': pct_dev,
            'adjusted_critical_t': crit_t,
            'bandwidth': bw})
    local_frames.append(pd.DataFrame({'COCNUM': gwr_gdf['COCNUM'].values,
                'predictor': pred,
                'local_coef': local_b,
                'local_se': local_se,
                'local_t': local_t,
                'ols_beta': ols_b,
                'same_sign': same_sign,
                'sig_category': np.where(sig_same,
                    'significant_same',
                    np.where(sig_rev,
                        'significant_reversal',
                        'nonsignificant')),
                'crit_t': crit_t}))

# Save both the predictor summary and detailed local classification records.
stability = pd.DataFrame(rows)
local_long = pd.concat(local_frames, ignore_index=True)
os.makedirs('outputs', exist_ok=True)
stability.to_csv('outputs/interpretation_stability_table.csv', index=False)
local_long.to_csv('outputs/gwr_local_stability_long.csv', index=False)
print('Saved: outputs/interpretation_stability_table.csv')
display(stability)
print('\nInterpretation: Significant_reversal_pct counts local sign reversals that also pass the adjusted critical t threshold.')
print('Same_sign_pct measures directional agreement with OLS; nonsignificant local estimates are reported separately.')
print(f"Maximum significant-reversal percentage: {stability['Significant_reversal_pct'].max():.2f}")


## Count and inspect all reversals — Corrections 1, 25–31, 35–39, 49

- For each of 141 CoCs and four predictors, compare local and OLS signs and calculate local t=coefficient/standard error.
- A significant reversal needs opposite direction AND |t| above the computed threshold.
- Opposite but unsupported signs are nonsignificant.
- These are coefficients, not residuals; no CoC is removed.

- The three supported-same/supported-reversal/nonsignificant categories partition the sample.
- Same-sign percentage is a separate overlapping measure.
- All percentages use 141 as denominator.
- The displayed qualifying rows answer exactly which seven Crime and three Tapestry results were counted.
- Count distinct places separately; records can overlap geographically.

## Step 41 — Name and count every significant reversal

**1. What this step does**

- Show the exact evidence behind seven Crime and three Tapestry reversals.

**2. Why we do it**

- A percentage alone cannot answer which places qualify and why.

**3. Process, in order**

1. Join CoC names to local records
2. Count all three categories for each predictor
3. Display qualifying rows and a supported/unsupported example; verify percentages

**4. How to read the output**

- Seven Crime CoCs and three Tapestry CoCs qualify; the other two predictors have none. Here these are ten distinct CoCs, confirmed from codes.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 41: Name and count every significant reversal
# Count the local categories and list the actual CoCs with significant reversals. A reversal
# needs both an opposite sign and a local absolute t-value above the adjusted cutoff. Verify
# that each predictor’s category counts total 141.

local_evidence = local_long.merge(merged_df[['COCNUM',
            'COCNAME']].drop_duplicates('COCNUM'),
    on='COCNUM',
    validate='many_to_one')

# Count significant-same, significant-reversal and nonsignificant areas for each predictor.
# Every area belongs to exactly one of these categories, so each row must total 141.
category_counts = pd.crosstab(local_evidence.predictor,
    local_evidence.sig_category).reindex(index=MODEL1_PREDICTORS,
    columns=['significant_same',
        'significant_reversal',
        'nonsignificant'],
    fill_value=0)
display(category_counts)
assert category_counts.sum(axis=1).eq(len(gwr_gdf)).all()

# List every supported reversal with coefficient, uncertainty, t-value and threshold.
reversals = local_evidence[local_evidence.sig_category.eq('significant_reversal')].copy()
reversals['Absolute t'] = reversals.local_t.abs()
display(reversals[['predictor',
            'COCNUM',
            'COCNAME',
            'ols_beta',
            'local_coef',
            'local_se',
            'local_t',
            'Absolute t',
            'crit_t']].sort_values(['predictor',
            'COCNUM']))
print('Reversal records:', len(reversals), 'Distinct CoCs:', reversals.COCNUM.nunique())
display(100 * category_counts / len(gwr_gdf))

# Demonstrate one qualifying result and compare it with an unsupported opposite sign.
crime = '2024 Total Crime Index'
example = local_evidence[local_evidence.predictor.eq(crime) & local_evidence.COCNUM.eq('NJ-506')].iloc[0]
print(f'NJ-506 Crime: OLS {example.ols_beta:.2f}; local {example.local_coef:.2f}; t {example.local_t:.4f}; threshold {example.crit_t:.4f}; classification {example.sig_category}.')
display(local_evidence[~local_evidence.same_sign & local_evidence.sig_category.eq('nonsignificant')][['predictor',
            'COCNUM',
            'ols_beta',
            'local_coef',
            'local_t',
            'crit_t']].head(1))
np.testing.assert_allclose(100 * category_counts.significant_reversal.values / len(gwr_gdf),
    stability.Significant_reversal_pct.values)


## Magnitude arithmetic — Corrections 26, 32, 37–39

- For each local coefficient calculate 100×|local−OLS|/|OLS|, then take the median of all 141 values.
- This is a coefficient comparison, not a percentage change in homelessness.
- Opposite signs can produce more than 100%.
- A positive Tapestry local median contrasts with its negative global coefficient, but support must still be tested individually.

## Step 42 — Explain local coefficient-magnitude differences

**1. What this step does**

- Reproduce the reported median percentage deviations.

**2. Why we do it**

- Coefficient differences are not percentage changes in homelessness.

**3. Process, in order**

1. Calculate each record's absolute relative difference from OLS
2. Take its median per predictor
3. Display a real example and verify the summary table

**4. How to read the output**

- The current medians are 33.45%, 74.19%, 63.33% and 172.08%. Opposite signs can produce values above 100%.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 42: Explain local coefficient-magnitude differences
# Measure how different each local coefficient is from its OLS reference in percentage terms,
# then take the median by predictor. This measures coefficient differences, not changes in the
# number of homeless people.

# Calculate 100 * |local - OLS| / |OLS| for each area, then take the middle value per predictor.
# Example: OLS=-100 and local=+50 give 150% deviation because their directions differ.
local_evidence['Absolute deviation %'] = 100 * (local_evidence.local_coef - local_evidence.ols_beta).abs() / local_evidence.ols_beta.abs()
deviations = local_evidence.groupby('predictor')['Absolute deviation %'].median().reindex(MODEL1_PREDICTORS)
display(deviations.to_frame('Median absolute deviation %'))
display(local_evidence[local_evidence.COCNUM.eq('NJ-506') & local_evidence.predictor.eq(crime)][['predictor',
            'COCNUM',
            'ols_beta',
            'local_coef',
            'Absolute deviation %']])
np.testing.assert_allclose(deviations.values, stability.Median_pct_deviation_from_OLS.values)
display(stability[['Predictor', 'OLS_beta', 'GWR_median', 'GWR_Q1', 'GWR_Q3', 'Same_sign_pct']])


## Simulation evidence retained — Corrections 14, 33

- The next cell runs 1,000 iterations using the installed mgwr algorithm, made observable so each simulated coefficient spread and selected bandwidth is retained.
- It uses the same seed, deepcopy, coordinate permutation, search/fit settings, population SD, and strict greater-than comparison.
- No cached simulation is substituted and no duplicate 1,000-fit run is needed.

- Outcome and predictors stay paired; coordinates move.
- Reselect bandwidth and refit each time.
- Count simulated spreads exceeding observed spread and divide by 1,000.
- This is mgwr's b/1,000 rule, distinct from Moran's plus-one rule.
- Zero exceedances are reported as <.001 following the paper's convention, not a precise population bound or zero probability.
- No extra cross-predictor adjustment is applied.

## Step 43 — Run the geographic-variability simulation

**1. What this step does**

- Test whether local coefficient spreads exceed random-location comparisons.

**2. Why we do it**

- Some variation occurs even when geography is randomly reassigned.

**3. Process, in order**

1. Measure observed coefficient standard deviations
2. Reassign coordinates 1,000 times; reselect bandwidth and refit each time
3. Count simulated spreads strictly greater than observed; save actual evidence and p-values

**4. How to read the output**

- Actual exceedances are 26, 414, 9 and 0 for the four predictors. Report zero as <.001 under the paper convention, not as zero probability; no extra cross-predictor adjustment is applied.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 43: Run the geographic-variability simulation
# Check whether observed local-coefficient variation exceeds variation under randomly reassigned
# locations. Repeat 1,000 times, selecting bandwidth and refitting each time. Retain every
# simulated spread and bandwidth so the p-values can be checked.

import os
import numpy as np
import pandas as pd
N_MC_ITERS = 1000
RANDOM_SEED = 5536
required = ['gwr_results', 'bw_selector', 'MODEL1_PREDICTORS']
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f'Run GWR bandwidth + fit cells first — missing: {missing}')
print(f'Running Monte Carlo spatial variability with n_iters={N_MC_ITERS} ...')
import copy
from tqdm.auto import tqdm

# Use temporary copies so shuffled fits do not replace the primary result.
temp_selector = copy.deepcopy(bw_selector)
temp_gwr = copy.deepcopy(gwr_results.model)
np.random.seed(RANDOM_SEED)

# Standard deviation measures how spread out the actual local coefficients are.
# Compare the same spread measure with each random-location refit.
observed_sd = np.std(gwr_results.params, axis=0)
simulated_sd = []
simulated_bw = []

# Randomly move the location labels while keeping each area’s inputs and outcome together.
# Example: area A keeps its data but receives area B’s coordinates.
# Choose bandwidth and refit GWR again for each of the 1,000 rearrangements.
for iteration in tqdm(range(N_MC_ITERS), desc='Geographic variability'):

    # Shuffle whole x/y pairs: coordinates move together while predictors and outcomes stay paired.
    shuffled_coords = np.random.permutation(gwr_results.model.coords)
    temp_selector.coords = shuffled_coords
    next_bw = temp_selector.search(**temp_selector.search_params)
    temp_gwr.bw = next_bw
    temp_gwr.coords = shuffled_coords
    shuffled_params = temp_gwr.fit(**temp_gwr.fit_params).params
    simulated_sd.append(np.std(shuffled_params, axis=0))
    simulated_bw.append(next_bw)
simulated_sd = np.asarray(simulated_sd)

# Count rearrangements with a coefficient spread greater than the actual spread.
# Example: 26 exceedances out of 1,000 give the library p-value .026.
exceedance_counts = (simulated_sd > observed_sd).sum(axis=0)
sv = exceedance_counts / float(N_MC_ITERS)
simulation_evidence = pd.DataFrame(simulated_sd, columns=['Intercept'] + MODEL1_PREDICTORS)
simulation_evidence.insert(0, 'Iteration', np.arange(1, N_MC_ITERS + 1))
simulation_evidence['Selected bandwidth'] = simulated_bw
simulation_evidence.to_csv('outputs/spatial_variability_simulations.csv', index=False)
display(pd.DataFrame({'Parameter': ['Intercept'] + MODEL1_PREDICTORS,
            'Observed SD': observed_sd,
            'Actual exceedances': exceedance_counts,
            'Raw p': sv}))
sv = list(sv)
assert len(sv) == 1 + len(MODEL1_PREDICTORS), f'Unexpected spatial_variability length: {len(sv)}'

# Save exact exceedances and report zero as <.001 under the paper convention.
mc_rows = []
for i, pred in enumerate(['Intercept'] + MODEL1_PREDICTORS):
    pval = float(sv[i])
    b = int(exceedance_counts[i])
    print(f'{pred}: p={pval:.6f}  (exceedances = {b}/{N_MC_ITERS})')
    if pred != 'Intercept':
        mc_rows.append({'Predictor': pred,
                'spatial_variability_p': pval,
                'spatial_variability_p_report': '<0.001' if pval == 0 else f'{pval:.3f}',
                'exceedances': b,
                'n_mc_iters': N_MC_ITERS,
                'seed': RANDOM_SEED})
mc_df = pd.DataFrame(mc_rows)
os.makedirs('outputs', exist_ok=True)
mc_df.to_csv('outputs/spatial_variability_pvalues.csv', index=False)
if 'stability' in globals():
    stability2 = stability.merge(mc_df, on='Predictor', how='left')
    stability2.to_csv('outputs/interpretation_stability_table.csv', index=False)
    print('\nUpdated: outputs/interpretation_stability_table.csv (with spatial_variability_p)')
    display(stability2[['Predictor', 'Same_sign_pct', 'Significant_reversal_pct', 'spatial_variability_p']])
else:
    print('Saved: outputs/spatial_variability_pvalues.csv')
    display(mc_df)


## Read the actual simulation distributions — Corrections 14, 33, 49

- The red line is the observed coefficient spread.
- Count the actual simulated spreads to its right and divide by 1,000.
- This tests the whole surface, not a reversal at any specific place.
- Read local classifications separately.

## Step 44 — Show the actual Monte Carlo distributions

**1. What this step does**

- Visualize the random-location comparisons used in the p-values.

**2. Why we do it**

- A plot helps explain how exceedance counting works.

**3. Process, in order**

1. Draw simulated spreads for each predictor
2. Place a red line at observed spread
3. Display the p-value table and verify actual counts

**4. How to read the output**

- Values to the right of the red line count as strict exceedances. This tests a whole coefficient surface, not a reversal at every location.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 44: Show the actual Monte Carlo distributions
# Plot the actual simulated coefficient spreads and mark the observed spread with a red line.
# Values to the right of that line count toward the variability p-value. This is a whole-surface
# test, separate from each area’s reversal classification.

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot actual simulated coefficient spreads and mark the observed spread in red.
for j, (ax, pred) in enumerate(zip(axes.ravel(), MODEL1_PREDICTORS), start=1):
    ax.hist(simulated_sd[:, j], bins=35, color='steelblue')
    ax.axvline(observed_sd[j], color='firebrick', label='Observed spread')
    ax.set_title(pred, fontsize=9)
    ax.set_xlabel('Local coefficient standard deviation')
    ax.set_ylabel('Iterations')
    ax.legend()
fig.tight_layout()
plt.show()
display(mc_df)
assert np.array_equal(mc_df.exceedances.values, exceedance_counts[1:])


## Step 45 — Save the compact analysis summary

**1. What this step does**

- Write the principal settings and global metrics to one small file.

**2. Why we do it**

- The basic result snapshot should be easy to inspect.

**3. Process, in order**

1. Collect n, predictor count and outcome
2. Record global fit, GWR bandwidth, critical t and iteration count
3. Save diagnostics.csv

**4. How to read the output**

- This compact file is not the full influence/assumption diagnostic analysis; those calculations follow.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 45: Save the compact analysis summary
# Save a compact snapshot of sample size, selected predictor count, OLS fit and GWR settings.
# This file supports result lookup; the more detailed influence and uncertainty checks follow.

import os
import pandas as pd
os.makedirs('outputs', exist_ok=True)
diag = pd.DataFrame({'metric': ['n_cocs',
            'n_predictors',
            'target',
            'R2',
            'AdjR2',
            'RMSE',
            'gwr_bandwidth',
            'gwr_critical_t',
            'n_mc_iters',
            'pit_target_used'],
        'value': [int(len(y_model)),
            int(len(MODEL1_PREDICTORS)),
            'Overall Homeless, 2021_pit',
            float(r2),
            float(adj_r2),
            float(rmse),
            float(bw),
            float(crit_t) if 'crit_t' in globals() else None,
            int(N_MC_ITERS) if 'N_MC_ITERS' in globals() else None,
            True]})
diag.to_csv('outputs/diagnostics.csv', index=False)
print('Saved: outputs/diagnostics.csv')
display(diag)


## Influence and uncertainty diagnostics — Corrections 45b, 45c, 40, 41

- Leverage measures unusual predictor combinations; Cook's distance measures influence on coefficients.
- Rank by CoC without automatically deleting an area.
- The NY-600 sensitivity comparison retains the same four predictors and original full-sample scaling, and refits after excluding that row.
- Its R² uses a different sample and is not a like-for-like forecast comparison.
- It does not reselect variables or change the baseline.

- The Koenker/Breusch–Pagan test checks constant residual variance.
- HC3 recalculates standard errors/p-values without changing coefficients.
- It does not correct spatial dependence or variable-selection uncertainty.
- Display conventional and HC3 evidence together; neither establishes causation.

## Step 46 — Test influence and uncertainty sensitivity

**1. What this step does**

- Calculate how the baseline depends on NY-600 and standard-error assumptions.

**2. Why we do it**

- Strong fit alone does not guarantee stable coefficients or significance.

**3. Process, in order**

1. Rank leverage and Cook's distance
2. Refit without NY-600 using the same predictors and original scaling
3. Run the variance diagnostic and compare conventional with HC3 uncertainty

**4. How to read the output**

- NY-600 strongly influences the fit; removing it changes two signs and gives R²≈.2882 on a different sample. HC3 p-values all exceed .05; HC3 does not correct spatial dependence or selection uncertainty.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 46: Test influence and uncertainty sensitivity
# Check how strongly particular CoCs affect the global equation and compare a fit excluding
# NY-600. Also test changing error variance and compare ordinary with HC3 uncertainty estimates.
# These diagnostic comparisons keep the reported baseline intact.

from statsmodels.stats.diagnostic import het_breuschpagan

# Find unusual input combinations (leverage) and areas that strongly affect the fit (Cook’s distance).
influence = ols_final.get_influence()
influence_evidence = prediction_evidence.copy()
influence_evidence['Leverage'] = influence.hat_matrix_diag
influence_evidence['Cooks distance'] = influence.cooks_distance[0]
display(influence_evidence.sort_values('Cooks distance', ascending=False).head(10))

# Refit after leaving out NY-600, using the same variables and original scaling.
# Compare the coefficients to see how much this one area influences the global reference.
keep = ~merged_df.loc[y_model.index, 'COCNUM'].eq('NY-600')
sensitivity_fit = OLS(y_model.loc[keep], add_constant(X_model_scaled.loc[keep])).fit()
sensitivity_coefficients = pd.DataFrame({'Predictor': MODEL1_PREDICTORS,
        'Full coefficient': ols_final.params.iloc[1:].values,
        'Without NY-600': sensitivity_fit.params.iloc[1:].values})
sensitivity_coefficients['Sign changed'] = np.sign(sensitivity_coefficients['Full coefficient']) != np.sign(sensitivity_coefficients['Without NY-600'])
display(sensitivity_coefficients)
display(pd.DataFrame({'Sample': ['Primary',
                'Without NY-600'],
            'n': [ols_final.nobs,
                sensitivity_fit.nobs],
            'R²': [ols_final.rsquared,
                sensitivity_fit.rsquared]}))

# Test whether error variability is constant, then compare ordinary and HC3 standard errors.
# HC3 keeps the coefficients but changes their uncertainty; it does not correct spatial dependence.

# Test the constant-error-variance assumption; a small diagnostic p-value questions that assumption.
bp = het_breuschpagan(ols_final.resid, ols_final.model.exog, robust=True)
display(pd.DataFrame({'Diagnostic': ['Koenker/BP LM', 'LM p', 'F', 'F p'], 'Value': bp}))

# Recalculate uncertainty with HC3 while keeping the fitted coefficients unchanged.
# Compare its p-values with the conventional ones; HC3 does not fix spatial dependence or selection.
hc3 = ols_final.get_robustcov_results(cov_type='HC3')
uncertainty = pd.DataFrame({'Parameter': ['Intercept'] + MODEL1_PREDICTORS,
        'Coefficient': ols_final.params.values,
        'Conventional SE': ols_final.bse.values,
        'Conventional p': ols_final.pvalues.values,
        'HC3 SE': hc3.bse,
        'HC3 p': hc3.pvalues})
display(uncertainty)


## Negative predictions and local collinearity — Corrections 45e, 45f

- Linear regression can produce negative fitted counts.
- List those areas and calculate the minimum; this limits unconstrained count forecasting, while observed counts remain unchanged.
- Alternative models require a separate agreed specification.

- Global VIF screening does not guarantee low overlap in every GWR neighborhood.
- The library computes local predictor VIFs and local condition numbers.
- Display maxima and responsible locations.
- Different diagnostics measure different aspects; one acceptable number does not erase another warning.

## Step 47 — Inspect impossible counts and local overlap

**1. What this step does**

- List negative OLS fitted values and local collinearity diagnostics.

**2. Why we do it**

- Practical count use and individual local coefficients need additional checks.

**3. Process, in order**

1. Select predictions below zero
2. Calculate local GWR VIFs and condition numbers
3. Display the largest values and their locations

**4. How to read the output**

- There are 42 negative predictions; minimum≈−4,872.73. Maximum local VIF≈26.09 and condition number≈21.03 measure different aspects of overlap.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 47: Inspect impossible counts and local overlap
# Find OLS estimates below zero, which are impossible as literal homelessness counts. Then
# measure predictor overlap inside GWR neighborhoods, because acceptable global VIFs do not
# guarantee acceptable local VIFs.

# List impossible fitted counts; these are predictions, not negative observed counts.
negative_predictions = prediction_evidence[prediction_evidence['OLS prediction'] < 0].sort_values('OLS prediction')
print('Negative counts:',
    len(negative_predictions),
    'Minimum fitted count:',
    prediction_evidence['OLS prediction'].min())
display(negative_predictions)

# Inspect predictor overlap within GWR neighborhoods rather than relying only on global VIFs.

# Calculate overlap separately inside local neighborhoods and identify the largest values.
local_corr, local_vifs, local_cn, local_vdp = gwr_results.local_collinearity()
vif_locations = pd.DataFrame(local_vifs, columns=MODEL1_PREDICTORS)
vif_locations.insert(0, 'COCNUM', gwr_gdf.COCNUM.values)
local_vif_long = vif_locations.melt(id_vars='COCNUM', var_name='Predictor', value_name='Local VIF')
condition_evidence = pd.DataFrame({'COCNUM': gwr_gdf.COCNUM.values, 'Local condition number': local_cn.ravel()})
display(local_vif_long.sort_values('Local VIF', ascending=False).head(10))
display(condition_evidence.sort_values('Local condition number', ascending=False).head(10))
print('Max local VIF:',
    local_vif_long['Local VIF'].max(),
    'Max condition number:',
    condition_evidence['Local condition number'].max())


## Generate Figure 5 from local classifications — Corrections 29–31, 49, 53

- Join current classifications by CoC code.
- Blue is supported in the global direction; red is supported opposite direction; gray within the sample is nonsignificant.
- Outside-study areas provide pale geographic context.
- Each panel has identical projected extent and equal aspect, with a white background and counts calculated from the same table.
- This image is generated from results, not manually colored.

## Step 48 — Generate the paper's stability-category map

**1. What this step does**

- Draw Figure 5 directly from current classifications.

**2. Why we do it**

- Map colors should be traceable to the same local records as the table.

**3. Process, in order**

1. Join categories by CoC code
2. Draw blue agreement, red reversal and gray nonsignificance
3. Use equal projected extents, pale outside-study context and a white background; save the figure

**4. How to read the output**

- This is a significance-category map, unlike the continuous coefficient maps. Gray within the sample is nonsignificant; pale outside-study areas are context.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 48: Generate the paper's stability-category map
# Draw Figure 5 using the saved local significance categories matched by CoC code. Blue means
# supported agreement, red means supported reversal and gray means nonsignificant. Use the same
# extent and proportions across panels.

from matplotlib.patches import Patch
fig, axes = plt.subplots(2, 2, figsize=(12, 13), facecolor='white')
colors = {'significant_same': '#287db3', 'significant_reversal': '#cb2a30', 'nonsignificant': '#b9b9b9'}

# Prepare the common study extent and geographic context for all panels.
background = hud.to_crs(ANALYSIS_CRS)
study = gwr_gdf[['COCNUM', 'geometry']].copy()
bounds = study.total_bounds
mx = (bounds[2] - bounds[0]) * 0.05
my = (bounds[3] - bounds[1]) * 0.05

# Join calculated categories by CoC and map supported agreement, reversal and nonsignificance.
for ax, pred in zip(axes.ravel(), MODEL1_PREDICTORS):
    background.plot(ax=ax, color='#ededed', edgecolor='white', linewidth=0.15)
    mapped = study.merge(local_evidence[local_evidence.predictor.eq(pred)][['COCNUM',
                'sig_category']],
        on='COCNUM',
        validate='one_to_one')
    assert mapped.sig_category.notna().all() and len(mapped) == len(study)
    mapped.plot(ax=ax, color=mapped.sig_category.map(colors), edgecolor='white', linewidth=0.3)
    ax.set_xlim(bounds[0] - mx, bounds[2] + mx)
    ax.set_ylim(bounds[1] - my, bounds[3] + my)
    ax.set_aspect('equal')
    ax.axis('off')
    c = category_counts.loc[pred]
    ax.set_title(pred + '\n' + f'Same: {c.significant_same} | Reversal: {c.significant_reversal} | Nonsignificant: {c.nonsignificant}',
        fontsize=9)

# Add the legend and save a white-background figure with equal map proportions.
fig.legend(handles=[Patch(color=color,
            label=label) for color,
        label in [('#287db3',
                'Significant same direction'),
            ('#cb2a30',
                'Significant reversal'),
            ('#b9b9b9',
                'Nonsignificant'),
            ('#ededed',
                'Outside study')]],
    loc='lower center',
    ncol=2)
fig.suptitle(f'Local interpretation stability | {len(study)} CoCs | bandwidth {bw:g} | critical |t| {crit_t:.4f}')
fig.tight_layout(rect=[0, 0.06, 1, 0.96])
fig.savefig('outputs/paper_figure5_stability.png', dpi=240, facecolor='white', bbox_inches='tight')
plt.show()


## ArcGIS figures and current input verification — Corrections 16, 43, 44, 51, 52

- Figures 3/4 are external ArcGIS Pro exports produced by `scripts/export_paper_arcgis_maps.py` in the licensed ArcGIS environment.
- This notebook provides the keyed counts and centroid export, displays the actual assets, and checks their recorded input hashes.
- Statistical notebook execution does not rebuild the APRX or fetch its basemap.
- A hash difference may be formatting: check keyed values before calling it a substantive data mismatch.

- Figure 3: 2024 Housing Affordability Index; Figure 4: 2022 ACS civilian veterans; yellow graduated symbols: 2021 PIT overall counts.
- The symbols use count classes rather than a continuous proportional-area scale.
- Outside-study gray areas are context.
- Housing Affordability is descriptive and is not in the primary four-predictor model.
- These maps show associations, not causes.

## Step 49 — Display external ArcGIS figures and provenance

**1. What this step does**

- Show Figures 3/4 and the recorded input-verification evidence.

**2. Why we do it**

- External maps must be distinguished from notebook statistical calculations.

**3. Process, in order**

1. Read the ArcGIS verification record
2. Display both existing map images
3. Calculate current input hashes for the final match check

**4. How to read the output**

- Figure 3 uses 2024 housing context; Figure 4 uses 2022 veterans; yellow classes show 2021 counts. The maps do not establish causation, and housing is not a primary predictor.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 49: Display external ArcGIS figures and provenance
# Display the existing ArcGIS Figures 3 and 4 and read their verification record. Calculate
# fingerprints of current input files for comparison with the recorded map inputs. This cell
# displays the exports; it does not rebuild the ArcGIS project.

import json, hashlib
from IPython.display import Image

# Read the external ArcGIS verification record and display the existing figure assets.
folder = Path('Paper/ArcGIS_Figures')
audit_path = folder / 'map_verification.json'
arcgis_audit = json.loads(audit_path.read_text(encoding='utf-8')) if audit_path.exists() else {}
print(json.dumps(arcgis_audit, indent=2))
for figure in [3, 4]:
    asset = folder / f'Figure_{figure}_ArcGIS.png'
    if asset.exists():
        display(Image(filename=str(asset), width=850))
    else:
        print('Missing external figure:', asset)

# Calculate a SHA256 fingerprint for each current input file.
# Matching fingerprints mean its bytes match the file recorded for the ArcGIS export.
for source in ['outputs/analysis_merged_141cocs.csv', 'outputs/arcgis_export/coc_spatial_verification.csv']:
    print('Current input SHA256:', source, hashlib.sha256(Path(source).read_bytes()).hexdigest())


## Source definitions and software provenance — Corrections 42, 45g–i, 50

- Appendix A is derived from the current contextual-field inventory and selection ledger.
- The Population Age 50+ aggregate replaces older age bands in the input.
- The truncated poverty category, Esri units/segment definitions, and boundary vintage still require authoritative source metadata.
- Do not invent descriptions from a truncated label.

- Versions and hashes identify this run, but do not certify survey completeness or source vintage.
- Bibliography correctness and in-text citation coverage require a separate citation audit.
- Software documentation pointers: [PySAL Moran](https://pysal.org/esda/stable/user-guide/global_morans_i.html), [mgwr results](https://mgwr.readthedocs.io/en/latest/generated/mgwr.gwr.GWRResults.html), [statsmodels diagnostics](https://www.statsmodels.org/stable/examples/notebooks/generated/regression_diagnostics.html).
- These links are method references; this code does not claim new online verification.

## Step 50 — Record software and input provenance

**1. What this step does**

- Identify the package versions and source snapshots used here.

**2. Why we do it**

- Reproduction depends on knowing both data and software.

**3. Process, in order**

1. Read installed package versions
2. Calculate input file hashes
3. Display them beside the outstanding metadata notes

**4. How to read the output**

- Boundary vintage, source units, the truncated poverty category and bibliography verification still need external evidence.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 50: Record software and input provenance
# Record package versions and source-file fingerprints to identify the software and data used.
# Fingerprints show which file bytes were used, but do not establish source definitions, count
# completeness or boundary vintage.

import importlib.metadata as metadata

# Record software versions and file fingerprints so this analysis can be traced to its inputs.
display(pd.DataFrame([{'Package': x,
                'Version': metadata.version(x)} for x in ['numpy',
                'pandas',
                'scipy',
                'statsmodels',
                'scikit-learn',
                'geopandas',
                'pyproj',
                'libpysal',
                'esda',
                'mgwr']]))
display(pd.DataFrame([{'Input': str(x),
                'SHA256': hashlib.sha256(x.read_bytes()).hexdigest()} for x in [Path('data/Final_Cleaned_Dataset_for_Homelessness_Modified.csv'),
                Path('data/PIT_dataset - 2021.csv'),
                Path(shp_path)]]))


## Paper evidence index and final reconciliation

- Repeated corrections refer to the same evidence in different paper paragraphs.
- The index names the calculation section for every numbered correction, including diagnostic subitems.
- Contribution/conclusion statements interpret these outputs; they are not additional experiments.
- Historical values/bandwidth 78 are not reconstructed controlled runs.

- The final table uses calculated values from this execution.
- Read diagnostics alongside fit.
- Successful execution means the workflow ran and evidence checks passed, not publication certification.
- External references and missing metadata retain their explicit status.

## Step 51 — Connect paper corrections to evidence

**1. What this step does**

- Provide a lookup table and one computed current-results summary.

**2. Why we do it**

- Every corrected number should have an identifiable calculation source.

**3. Process, in order**

1. List each correction's notebook section
2. Collect current computed values
3. Check unique local records and complete categories; save the result summary

**4. How to read the output**

- Use this index during presentation questions. Several corrections refer to the same underlying result; successful execution is not publication certification.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


In [ ]:
# Step 51: Connect paper corrections to evidence
# Link numbered paper corrections to the notebook sections deriving their evidence. Collect the
# current calculated values into a summary file and check unique local records and complete
# category labels.

# Connect each paper correction to the section deriving its evidence.
evidence_index = pd.DataFrame([('3,5,6,15,42,50',
            'Input inventory; selection ledger; provenance'),
        ('7,9',
            'Matched outcomes; exclusions; VIF; p-value selection'),
        ('1,17,18,19,46,47',
            'Primary coefficients; worked fit; five-outcome performance'),
        ('8,45f',
            'Projection and geometry audit'),
        ('10,20,21,22,23,48',
            'Moran evidence; distance bands; OLS/GWR comparison'),
        ('11,12,13,24,27',
            'GWR neighborhood and adjusted threshold'),
        ('1,25,26,28,30,31,35,36,37,38,39,49',
            'Count and inspect reversals; stability'),
        ('26,32,37,38,39,49',
            'Magnitude arithmetic'),
        ('14,33,34,49',
            'Simulation evidence and distributions'),
        ('45b,45c,45d,45e,45f',
            'Scaling; influence; HC3; negative predictions; local collinearity'),
        ('16,43,44,51,52',
            'External ArcGIS assets and verification'),
        ('29,30,31,53',
            'Generated Figure 5'),
        ('2,4,34,35,36,40,41',
            'Combined stability and diagnostic interpretation'),
        ('45a',
            'Diagnostic-section organization; no independent numerical claim'),
        ('45g,45h,45i',
            'Software references and external metadata still needed')],
    columns=['Paper corrections',
        'Evidence section'])
display(evidence_index)

# Collect current calculated results and check unique local records and complete categories.
paper_values = pd.DataFrame({'Metric': ['CoCs',
            'Jurisdictions',
            'Predictors',
            'OLS R²',
            'OLS adjusted R²',
            'OLS RMSE',
            'GWR R²',
            'GWR RMSE',
            'GWR AICc',
            'Bandwidth',
            'Critical t',
            'OLS Moran I',
            'OLS p_sim',
            'GWR Moran I',
            'GWR p_sim',
            'Crime reversals',
            'Tapestry reversals',
            'Negative predictions',
            'Max local VIF',
            'Max local condition number'],
        'Value': [len(y_model),
            df_enriched.ST.nunique(),
            len(MODEL1_PREDICTORS),
            r2,
            adj_r2,
            rmse,
            gwr_r2,
            gwr_rmse,
            gwr_results.aicc,
            bw,
            crit_t,
            mi_res.I,
            mi_res.p_sim,
            mi_gwr_resid.I,
            mi_gwr_resid.p_sim,
            category_counts.loc['2024 Total Crime Index',
                'significant_reversal'],
            category_counts.loc['2024 Pop 18+ in Tapestry Seg 12D',
                'significant_reversal'],
            len(negative_predictions),
            local_vif_long['Local VIF'].max(),
            condition_evidence['Local condition number'].max()]})
display(paper_values)
paper_values.to_csv('outputs/paper_results_evidence.csv', index=False)
assert len(local_evidence) == len(gwr_gdf) * len(MODEL1_PREDICTORS)
assert not local_evidence[['COCNUM', 'predictor']].duplicated().any()
assert set(local_evidence.sig_category) == {'significant_same', 'significant_reversal', 'nonsignificant'}
print('Reconciliation passed: unique records, complete categories, matched scaling and percentages checked. External metadata/citation checks remain separate.')


## Execution verification and ArcGIS input match

- This final check reads the actual saved output files, checks their agreement, and compares the ArcGIS export's recorded source hashes with the files generated by this run.
- Identical hashes establish the same input bytes as the external map export; they do not rerender the ArcGIS project or reverify citation metadata.
- The result is shown explicitly rather than leaving two hashes for a reader to compare manually.

- The run summary checks that all earlier code cells have saved execution outputs and no recorded errors.
- It is evidence of completion, not a claim of causal or forecasting validity.
- Read the influence, uncertainty and local-collinearity diagnostics before interpreting the study.

## Step 52 — Verify saved outputs and ArcGIS input matches

**1. What this step does**

- Check the actual saved evidence files after execution.

**2. Why we do it**

- A readable notebook should also reconcile its outputs and report completion honestly.

**3. Process, in order**

1. Compare recorded ArcGIS input hashes with current files
2. Check 564 local rows and 1,000 simulations, percentages and exceedance arithmetic
3. Read saved notebook execution records and check for cell errors

**4. How to read the output**

- The saved evidence agrees and all code cells have execution outputs. Identical map-input hashes verify input snapshots, not a newly rerendered APRX or citation audit.


- **Reading the code:** follow the numbered workflow and the comments directly inside the next cell.


## Historical archived-run audit
This original cell checked cached ArcGIS hashes and saved execution counts. Outputs were cleared for public release, so that archived-state assertion is intentionally not rerun here. Inspect fresh numerical outputs and save the executed notebook separately. The original source remains unchanged.
